# Who's to Blame? Actual Causation and Congestion Pricing for Explaining Macro-Placement Failures

**End-to-end reproduction notebook.** Runs top-to-bottom on a Kaggle CPU or GPU kernel with no
internet access and no external datasets. Everything below â€” designs, the interventional oracle,
the surrogates, the attributions, the closed-loop validation, and every figure â€” is produced here.

---

### The argument

Saliency asks *"what did the model look at?"* â€” a correlational question about a predictor.
A designer asks the courtroom question: **"but for macro M3 being here, would this DRC hotspot exist?"**
That is a counterfactual about the *design*, not about the network, and the Halpernâ€“Pearl theory of
actual causation gives it a precise form. We compute, per macro:

| quantity | reading |
| --- | --- |
| **PN** â€” probability of necessity | move it away: does the violation vanish? |
| **PS** â€” probability of sufficiency | put it back into a clean floorplan: does the violation appear? |
| **PNS** | necessary *and* sufficient |
| **HP responsibility** (Chocklerâ€“Halpern) | how small a coalition of other changes it needs to become pivotal â€” $1/(1+|W|)$ |
| **Blame** | responsibility averaged over the designer's uncertainty about the context |
| **Congestion price** (Vickrey/Pigou) | the routing cost this macro imposes *on everybody else* â€” a shadow price in cost units |
| **Shapley value** | the game-theoretic split of the total DRC count across macros |

The trick that makes this cheap and defensible: **we own a perfect interventional oracle** â€” the
place-and-route engine itself. We run a few hundred *real* interventions (move one macro, swap two,
jitter a coalition), fit a surrogate (U-Net on congestion maps + a GNN on the macro graph) to
amortize the $2^{n}$ counterfactual queries, and then **verify the top attributions with real
oracle reruns**. That loop is the contribution.

### What the notebook produces

1. An interventional dataset over several designs (`interventions.csv`).
2. Two trained surrogates with held-out fidelity numbers.
3. Six attribution methods on the same designs: HP-responsibility, blame, Pigouvian price,
   Shapley, Grad-CAM, input-gradient saliency.
4. **The paper table**: responsibility-ranked edits vs. saliency-ranked edits, DRC reduction measured
   by re-running the oracle. Plus deletion/insertion faithfulness curves and rank correlation to
   exhaustive ground-truth necessity.
5. A figure pack: matplotlib PDFs, PIL-rendered die shots, hand-written SVG vector figures, an
   animated GIF of the repair loop, interactive Plotly HTML, and a self-contained `report.html`.

### A note on the oracle

Kaggle kernels have no OpenROAD/ORFS install and no network. So the notebook ships a
**self-contained mini-EDA oracle**: an ePlace-style analytical global placer for the standard-cell
clusters that *reacts* to macro positions, followed by a PathFinder/FastRoute-style
negotiated-congestion global router with rip-up-and-reroute on a GCell grid, whose overflow map is
the DRC-hotspot proxy. It is a real placer and a real router, just small. Every experiment here is
oracle-agnostic: `Oracle.evaluate(placement) -> Result` is the only interface the causal machinery
touches, and the final section ships a drop-in `ORFSOracle` adapter that shells out to `openroad`
on a machine that has it, so the identical notebook reproduces against ibex/aes on Nangate45
outside Kaggle.


In [ ]:
# =====================================================================================
# 1. Environment, configuration, reproducibility
# =====================================================================================
import os, sys, json, math, time, itertools, random, hashlib, textwrap, warnings
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Tuple, Optional, Sequence

import numpy as np

warnings.filterwarnings("ignore")

# ---- output directory (works on Kaggle and locally) ---------------------------------
OUT = "/kaggle/working/outputs" if os.path.isdir("/kaggle/working") else "outputs"
for sub in ("figures", "images", "svg", "html", "tables", "data", "anim"):
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)
P = lambda *a: os.path.join(OUT, *a)

# ---- global scale switch -------------------------------------------------------------
# QUICK=True   -> ~8-12 min end-to-end on a Kaggle CPU kernel (smoke test, all outputs produced)
# QUICK=False  -> ~1.5-3 h, the numbers reported in the paper
QUICK = bool(int(os.environ.get("MB_QUICK", "1")))


@dataclass
class Config:
    seed: int = 20260815
    grid: int = 48                 # GCell grid is grid x grid
    die: float = 1000.0            # die edge, um
    # --- oracle fidelity ---
    cap_k: float = 1.0             # track supply = cap_k * cap_pct-quantile of unconstrained demand
    cap_pct: float = 97.0
    place_iters: int = 90          # ePlace-lite gradient steps
    route_iters: int = 6           # negotiated-congestion rip-up & reroute rounds
    # --- experiment sizes (overridden by QUICK below) ---
    n_interventions: int = 420     # real oracle interventions per design (surrogate training set)
    shapley_perms: int = 4000      # Monte-Carlo permutations (surrogate-evaluated)
    resp_max_witness: int = 3      # max |W| searched for HP responsibility
    unet_epochs: int = 120
    gnn_epochs: int = 400
    verify_topk: int = 4           # attributions re-verified with the real oracle
    repair_topk: int = 3           # macros edited in the closed-loop repair experiment
    repair_candidates: int = 12    # legal sites tried per edited macro

    def quicken(self):
        self.grid = 40
        self.place_iters = 55
        self.route_iters = 4
        self.n_interventions = 130
        self.shapley_perms = 900
        self.resp_max_witness = 2
        self.unet_epochs = 35
        self.gnn_epochs = 150
        self.verify_topk = 3
        self.repair_topk = 2
        self.repair_candidates = 6
        return self


CFG = Config()
if QUICK:
    CFG.quicken()

random.seed(CFG.seed)
np.random.seed(CFG.seed % (2**31))

# ---- optional deps -------------------------------------------------------------------
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    torch.manual_seed(CFG.seed)
    DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    HAS_TORCH = True
except Exception as e:                                    # pragma: no cover
    HAS_TORCH, DEV = False, None
    print("torch unavailable -> surrogates fall back to numpy ridge models:", e)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from PIL import Image, ImageDraw, ImageFont, ImageFilter

try:
    import plotly.graph_objects as go
    import plotly.io as pio
    HAS_PLOTLY = True
except Exception:                                          # pragma: no cover
    HAS_PLOTLY = False

import pandas as pd

print(f"outputs -> {os.path.abspath(OUT)}")
print(f"QUICK={QUICK}  torch={HAS_TORCH} ({DEV})  plotly={HAS_PLOTLY}  grid={CFG.grid}")

## 1. Designs

Three blocks in the shape of small ORFS designs: hard macros (SRAMs), standard-cell clusters grouped into modules, a clustered hypergraph netlist and fixed IO pads. Each carries two reference floorplans — the **as-given** one we are asked to explain, and a canonical **peripheral** one that serves as the default `off` setting for every counterfactual.

In [ ]:
# =====================================================================================
# 2. Designs: synthetic-but-structured netlists in the shape of small ORFS blocks
# =====================================================================================
# A design carries: hard macros (SRAMs), standard-cell clusters grouped into modules,
# a clustered hypergraph netlist, and fixed IO pads. Node index space:
#     [0, NC)                cell clusters      (movable, placed by the oracle's placer)
#     [NC, NC+NM)            macros             (fixed by us -- the decision variables)
#     [NC+NM, NC+NM+NIO)     IO pads            (fixed on the die boundary)

@dataclass
class Design:
    name: str
    die: float
    macro_wh: np.ndarray        # (NM, 2)  macro widths/heights
    macro_names: List[str]
    base_sites: np.ndarray      # (NM, 2)  centres of the *actual* (as-given) floorplan
    null_sites: np.ndarray      # (NM, 2)  centres of the canonical peripheral "off" floorplan
    cell_area: np.ndarray       # (NC,)
    cell_module: np.ndarray     # (NC,)
    macro_module: np.ndarray    # (NM,)
    net_ptr: np.ndarray         # (NNET+1,) CSR pointer into net_nodes
    net_nodes: np.ndarray       # flat node ids
    io_xy: np.ndarray           # (NIO, 2)
    n_modules: int

    @property
    def NM(self): return len(self.macro_wh)
    @property
    def NC(self): return len(self.cell_area)
    @property
    def NIO(self): return len(self.io_xy)
    @property
    def NNET(self): return len(self.net_ptr) - 1


def _pack_periphery(macro_wh: np.ndarray, die: float, pad: float = 12.0) -> np.ndarray:
    """Canonical 'off' floorplan: macros hug the die boundary, largest first, walking the
    perimeter. This is what a human does with SRAMs when they are not fighting for area, and
    it is the reference (do(M = off)) setting for every counterfactual below."""
    order = np.argsort(-macro_wh.max(axis=1))
    sites = np.zeros_like(macro_wh)
    cursor, side = pad, 0                       # side: 0 bottom, 1 right, 2 top, 3 left
    for i in order:
        w, h = macro_wh[i]
        span = w if side % 2 == 0 else h
        depth = h if side % 2 == 0 else w
        if cursor + span + pad > die:           # move to the next edge
            side, cursor = (side + 1) % 4, pad
        c = cursor + span / 2
        d = pad + depth / 2
        sites[i] = {0: (c, d), 1: (die - d, c), 2: (die - c, die - d), 3: (d, die - c)}[side]
        cursor += span + pad
    return sites


def _random_legal_sites(rng, macro_wh, die, margin=25.0, tries=4000) -> np.ndarray:
    """A plausible but imperfect as-given floorplan: macros dropped in the core with a
    non-overlap constraint and a mild bias toward the centre, which is exactly the habit that
    manufactures congestion hotspots."""
    sites = np.zeros_like(macro_wh)
    placed = []
    for i in np.argsort(-macro_wh.prod(axis=1)):
        w, h = macro_wh[i]
        for _ in range(tries):
            cx = np.clip(rng.normal(die / 2, die / 5.5), margin + w / 2, die - margin - w / 2)
            cy = np.clip(rng.normal(die / 2, die / 5.5), margin + h / 2, die - margin - h / 2)
            box = (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)
            if all(not (box[0] < b[2] + 8 and b[0] - 8 < box[2] and
                        box[1] < b[3] + 8 and b[1] - 8 < box[3]) for b in placed):
                placed.append(box); sites[i] = (cx, cy); break
        else:                                    # give up: park it at the periphery
            sites[i] = _pack_periphery(macro_wh, die)[i]
            placed.append((sites[i, 0] - w / 2, sites[i, 1] - h / 2,
                           sites[i, 0] + w / 2, sites[i, 1] + h / 2))
    return sites


def make_design(name: str, n_macros: int, n_cells: int, n_nets: int, n_modules: int,
                die: float, seed: int) -> Design:
    rng = np.random.default_rng(seed)
    macro_wh = rng.uniform(0.075, 0.155, size=(n_macros, 2)) * die
    macro_wh[:, 1] *= rng.uniform(0.7, 1.4, size=n_macros)          # SRAMs are not square
    macro_wh = np.clip(macro_wh, 0.05 * die, 0.20 * die)
    macro_names = [f"M{i}" for i in range(n_macros)]

    cell_module = rng.integers(0, n_modules, size=n_cells)
    macro_module = rng.integers(0, n_modules, size=n_macros)        # each SRAM serves a module
    cell_area = rng.lognormal(0.0, 0.5, size=n_cells) * (0.45 * die * die / n_cells)

    NC, NM = n_cells, n_macros
    n_io = max(16, n_modules * 4)
    t = np.linspace(0, 4, n_io, endpoint=False)
    side, frac = np.floor(t).astype(int), t - np.floor(t)
    io_xy = np.stack([np.where(side == 0, frac * die, np.where(side == 1, die,
                      np.where(side == 2, (1 - frac) * die, 0.0))),
                      np.where(side == 0, 0.0, np.where(side == 1, frac * die,
                      np.where(side == 2, die, (1 - frac) * die)))], axis=1)

    # ---- clustered hypergraph: mostly intra-module, macros bound to their module ---------
    by_mod = [np.where(cell_module == m)[0] for m in range(n_modules)]
    macros_of = [np.where(macro_module == m)[0] for m in range(n_modules)]
    ptr, nodes = [0], []
    for _ in range(n_nets):
        m = int(rng.integers(0, n_modules))
        deg = int(np.clip(rng.lognormal(0.85, 0.55), 2, 7))
        pool = by_mod[m] if len(by_mod[m]) >= deg else np.arange(NC)
        pins = list(rng.choice(pool, size=min(deg, len(pool)), replace=False))
        if rng.random() < 0.14:                                     # cross-module net
            m2 = int(rng.integers(0, n_modules))
            if len(by_mod[m2]): pins.append(int(rng.choice(by_mod[m2])))
        if len(macros_of[m]) and rng.random() < 0.34:               # memory access net
            pins.append(NC + int(rng.choice(macros_of[m])))
        elif rng.random() < 0.05:
            pins.append(NC + int(rng.integers(0, NM)))
        if rng.random() < 0.08:                                     # primary IO
            pins.append(NC + NM + int(rng.integers(0, n_io)))
        pins = list(dict.fromkeys(pins))
        if len(pins) < 2: continue
        nodes.extend(pins); ptr.append(len(nodes))

    # standard-cell cluster area tracks its pin count (as it does in a real netlist); this is what
    # makes uniform-density placement also mean uniform *pin* density, hence smooth routing demand
    nodes_arr = np.asarray(nodes, dtype=np.int64)
    pin_cnt = np.bincount(nodes_arr[nodes_arr < NC], minlength=NC).astype(float)
    cell_area = cell_area * (0.4 + pin_cnt) / (0.4 + pin_cnt).mean()

    return Design(name=name, die=die, macro_wh=macro_wh, macro_names=macro_names,
                  base_sites=_random_legal_sites(rng, macro_wh, die),
                  null_sites=_pack_periphery(macro_wh, die),
                  cell_area=cell_area, cell_module=cell_module, macro_module=macro_module,
                  net_ptr=np.asarray(ptr), net_nodes=np.asarray(nodes, dtype=np.int64),
                  io_xy=io_xy, n_modules=n_modules)


# Sized after the small ORFS blocks the method is meant for (ibex / aes / jpeg on Nangate45).
_SPECS = [
    ("ibex_like",  8, 3000, 4600, 7, 1000.0),
    ("aes_like",   6, 2400, 3800, 5,  860.0),
    ("jpeg_like", 11, 3800, 5800, 9, 1180.0),
]
if QUICK:
    _SPECS = [(n, m, int(c * .55), int(nn * .55), k, d) for (n, m, c, nn, k, d) in _SPECS]

DESIGNS: Dict[str, Design] = {
    s[0]: make_design(s[0], s[1], s[2], s[3], s[4], s[5], CFG.seed + 17 * i)
    for i, s in enumerate(_SPECS)
}
for d in DESIGNS.values():
    print(f"{d.name:11s} macros={d.NM:3d} cell-clusters={d.NC:5d} nets={d.NNET:5d} "
          f"pins={len(d.net_nodes):6d} modules={d.n_modules}  die={d.die:.0f}um")

## 2. The interventional oracle

An ePlace-style analytical placer that re-places the standard cells in response to the macros, followed by a PathFinder-style negotiated-congestion global router. `evaluate(macro_xy) -> Result` is the only interface the causal machinery ever touches — swap in the OpenROAD adapter at the end of the notebook and every experiment below reruns unchanged.

In [ ]:
# =====================================================================================
# 3. The interventional oracle: an ePlace-style placer + a PathFinder-style global router
# =====================================================================================
# Oracle.evaluate(macro_xy) -> Result is the ONLY interface the causal machinery uses.
# Swap in ORFSOracle (final section) to run the identical experiments against OpenROAD.

@dataclass
class Result:
    drc: float                  # DRC-hotspot proxy = total positive routing overflow
    n_hotspot: int              # GCells whose overflow crosses the violation threshold
    wl: float                   # routed wirelength (GCell units)
    hotspot: np.ndarray         # (G,G) per-GCell violation map  -- the "DRC marker" layer
    ovfl: np.ndarray            # (G,G) continuous overflow (h+v)
    usage: np.ndarray           # (G,G) total usage
    cap: np.ndarray             # (G,G) total capacity
    lam: np.ndarray             # (G,G) PathFinder history cost == estimated capacity duals
    cell_xy: np.ndarray         # (NC,2) placer output -- macros move, cells follow
    net_cost: np.ndarray        # (NNET,) per-net routed cost, for the externality accounting
    macro_xy: np.ndarray


def _box_blur(a, r):
    if r <= 0: return a
    k = 2 * r + 1
    c = np.cumsum(np.pad(a, ((r + 1, r), (0, 0)), mode="edge"), axis=0)
    a = (c[k:] - c[:-k]) / k
    c = np.cumsum(np.pad(a, ((0, 0), (r + 1, r)), mode="edge"), axis=1)
    return (c[:, k:] - c[:, :-k]) / k


def _gauss(a, r):
    return _box_blur(_box_blur(_box_blur(a, r), r), r)


class Oracle:
    """Deterministic function of the macro placement. Everything downstream -- the standard-cell
    placement, the routing topology, the congestion -- is recomputed from scratch, which is what
    makes an intervention here a genuine do() on the design rather than a perturbation of an input
    tensor."""

    def __init__(self, design: Design, cfg: Config):
        self.d, self.cfg = design, cfg
        self.G = cfg.grid
        self.gs = design.die / self.G                     # GCell size
        rng = np.random.default_rng(CFG.seed ^ hash(design.name) % (2**31))
        # deterministic starting point for the placer: identical for every intervention, so any
        # difference in the final placement is *caused* by the macro move
        self.cell_init = rng.uniform(0.12, 0.88, size=(design.NC, 2)) * design.die
        self.macro_pin_off = np.array([[.5, 0], [0, .5], [-.5, 0], [0, -.5]])
        self.calls = 0
        self._precompute_nets()
        self._calibrate()

    def _calibrate(self):
        """Track capacity is a technology constant, not a function of the placement, so we fix it
        once: route the canonical peripheral floorplan with unlimited capacity and set the per-GCell
        supply from a high quantile of that demand. Every intervention then sees the same supply."""
        self.base_cap = 1e9
        r = self.route(self.place(self.d.null_sites), self.d.null_sites, seed=0)
        u = np.concatenate([r["uh"].ravel(), r["uv"].ravel()])
        self.base_cap = float(max(4.0, self.cfg.cap_k * np.percentile(u, self.cfg.cap_pct)))

    # -- netlist bookkeeping -----------------------------------------------------------
    def _precompute_nets(self):
        d = self.d
        ptr, nodes = d.net_ptr, d.net_nodes
        self.pin_net = np.repeat(np.arange(d.NNET), np.diff(ptr))
        self.pin_node = nodes
        self.pin_is_cell = nodes < d.NC
        self.pin_is_macro = (nodes >= d.NC) & (nodes < d.NC + d.NM)
        self.pin_is_io = nodes >= d.NC + d.NM
        self.pin_macro_id = np.where(self.pin_is_macro, nodes - d.NC, 0)
        self.pin_io_id = np.where(self.pin_is_io, nodes - d.NC - d.NM, 0)
        self.pin_cell_id = np.where(self.pin_is_cell, nodes, 0)
        self.pin_macro_face = np.arange(len(nodes)) % 4
        self.net_deg = np.diff(ptr)
        # nets touching each macro -- needed for the "cost to everyone else" accounting
        self.nets_of_macro = [np.unique(self.pin_net[self.pin_is_macro & (self.pin_macro_id == m)])
                              for m in range(d.NM)]
        # cell -> nets incidence for the wirelength force
        self.cell_pin_idx = np.where(self.pin_is_cell)[0]

    # -- macro geometry ----------------------------------------------------------------
    def macro_density(self, macro_xy):
        """Fractional GCell coverage by hard macros (exact box/GCell overlap area)."""
        d, G, gs = self.d, self.G, self.gs
        cov = np.zeros((G, G))
        edges = np.arange(G + 1) * gs
        for i in range(d.NM):
            w, h = d.macro_wh[i]
            x0, x1 = macro_xy[i, 0] - w / 2, macro_xy[i, 0] + w / 2
            y0, y1 = macro_xy[i, 1] - h / 2, macro_xy[i, 1] + h / 2
            ox = np.clip(np.minimum(edges[1:], x1) - np.maximum(edges[:-1], x0), 0, None)
            oy = np.clip(np.minimum(edges[1:], y1) - np.maximum(edges[:-1], y0), 0, None)
            cov += np.outer(oy, ox) / (gs * gs)
        return np.clip(cov, 0, 1)

    def macro_cover_one(self, macro_xy, m):
        d, G, gs = self.d, self.G, self.gs
        edges = np.arange(G + 1) * gs
        w, h = d.macro_wh[m]
        x0, x1 = macro_xy[m, 0] - w / 2, macro_xy[m, 0] + w / 2
        y0, y1 = macro_xy[m, 1] - h / 2, macro_xy[m, 1] + h / 2
        ox = np.clip(np.minimum(edges[1:], x1) - np.maximum(edges[:-1], x0), 0, None)
        oy = np.clip(np.minimum(edges[1:], y1) - np.maximum(edges[:-1], y0), 0, None)
        return np.clip(np.outer(oy, ox) / (gs * gs), 0, 1)

    # -- stage 1: analytical placement of the standard-cell clusters --------------------
    def _shift(self, xy, area, mcov, axis, nb=14, blend=0.55):
        """Bin-based cell shifting (Gordian-L / Kraftwerk style): inside each slice, remap the
        cells monotonically so that they fill the *free* area uniformly. Macro coverage removes
        free area, so cells flow around hard blockages instead of piling on top of them."""
        d, G = self.d, self.G
        die = d.die
        oth = 1 - axis
        edges = np.linspace(0, die, G + 1)
        occ_free = 1.0 - (mcov if axis == 0 else mcov.T)      # rows indexed by the other axis
        sl = np.clip((xy[:, oth] / die * nb).astype(int), 0, nb - 1)
        rows_per = G / nb
        new = xy[:, axis].copy()
        for j in range(nb):
            idx = np.where(sl == j)[0]
            if len(idx) < 4: continue
            r0, r1 = int(j * rows_per), max(int(j * rows_per) + 1, int((j + 1) * rows_per))
            free = occ_free[r0:r1].sum(axis=0)                # (G,) free width per bin
            cfree = np.concatenate([[0.0], np.cumsum(np.maximum(free, 1e-3))])
            order = idx[np.argsort(xy[idx, axis])]
            ca = np.cumsum(area[order]); ca = (ca - 0.5 * area[order]) / ca[-1]
            new[order] = np.interp(ca * cfree[-1], cfree, edges)
        xy[:, axis] = (1 - blend) * xy[:, axis] + blend * new
        return xy

    def place(self, macro_xy):
        """ePlace-lite: star-model wirelength force + a density force, with periodic bin-based
        shifting to keep the standard-cell density uniform over the free area. Hard macros enter
        as fixed charge, so moving a macro genuinely re-places the whole block."""
        d, cfg, G, gs = self.d, self.cfg, self.G, self.gs
        xy = self.cell_init.copy()
        area = d.cell_area / d.cell_area.mean()
        mcov = self.macro_density(macro_xy)
        io = d.io_xy
        pn, pnode = self.pin_net, self.pin_node
        v = np.zeros_like(xy)
        for it in range(cfg.place_iters):
            # ---- wirelength: pull every pin toward its net centroid (star model) --------
            pos = np.empty((len(pnode), 2))
            pos[self.pin_is_cell] = xy[self.pin_cell_id[self.pin_is_cell]]
            pos[self.pin_is_macro] = (macro_xy[self.pin_macro_id[self.pin_is_macro]] +
                                      self.macro_pin_off[self.pin_macro_face[self.pin_is_macro]] *
                                      d.macro_wh[self.pin_macro_id[self.pin_is_macro]])
            pos[self.pin_is_io] = io[self.pin_io_id[self.pin_is_io]]
            cen = np.zeros((d.NNET, 2)); np.add.at(cen, pn, pos)
            cen /= self.net_deg[:, None]
            gwl = np.zeros_like(xy)
            ci = self.cell_pin_idx
            np.add.at(gwl, self.pin_cell_id[ci], pos[ci] - cen[pn[ci]])
            # ---- density: push down the gradient of the blurred occupancy field ---------
            gx = np.clip((xy[:, 0] / gs).astype(int), 0, G - 1)
            gy = np.clip((xy[:, 1] / gs).astype(int), 0, G - 1)
            dens = np.zeros((G, G)); np.add.at(dens, (gy, gx), area)
            dens = dens / max(dens.mean(), 1e-9) + 8.0 * mcov     # macros = large fixed charge
            phi = _gauss(dens, max(1, G // 12))
            fy, fx = np.gradient(phi)
            g = 0.05 * gwl + 0.9 * gs * np.stack([fx[gy, gx], fy[gy, gx]], 1)
            v = 0.7 * v - 0.5 * g
            xy = np.clip(xy + v, 2.0, d.die - 2.0)
            if it % 5 == 4 and it < cfg.place_iters - 2:          # keep density uniform
                xy = self._shift(xy, area, mcov, it // 5 % 2)
                xy = np.clip(xy, 2.0, d.die - 2.0)
        # ---- hard-blockage cleanup: no standard cell may sit inside a macro -------------
        for i in range(d.NM):
            w, h = d.macro_wh[i]; cx, cy = macro_xy[i]
            dx, dy = xy[:, 0] - cx, xy[:, 1] - cy
            inside = (np.abs(dx) < w / 2) & (np.abs(dy) < h / 2)
            if inside.any():
                px = (w / 2 - np.abs(dx[inside])); py = (h / 2 - np.abs(dy[inside]))
                usex = px < py
                ii = np.where(inside)[0]
                xy[ii[usex], 0] = cx + np.sign(dx[inside][usex] + 1e-9) * (w / 2 + 1.5)
                xy[ii[~usex], 1] = cy + np.sign(dy[inside][~usex] + 1e-9) * (h / 2 + 1.5)
        return np.clip(xy, 1.0, d.die - 1.0)

    # -- stage 2: negotiated-congestion global routing ---------------------------------
    def _segments(self, cell_xy, macro_xy):
        """Pins -> GCell coords -> 2-pin segments by chaining each net's pins in x-major order
        (a cheap single-trunk RSMT that FastRoute-class routers use as their starting topology)."""
        d, G, gs = self.d, self.G, self.gs
        pos = np.empty((len(self.pin_node), 2))
        pos[self.pin_is_cell] = cell_xy[self.pin_cell_id[self.pin_is_cell]]
        pos[self.pin_is_macro] = (macro_xy[self.pin_macro_id[self.pin_is_macro]] +
                                  self.macro_pin_off[self.pin_macro_face[self.pin_is_macro]] *
                                  d.macro_wh[self.pin_macro_id[self.pin_is_macro]])
        pos[self.pin_is_io] = d.io_xy[self.pin_io_id[self.pin_is_io]]
        gx = np.clip((pos[:, 0] / gs).astype(np.int64), 0, G - 1)
        gy = np.clip((pos[:, 1] / gs).astype(np.int64), 0, G - 1)
        order = np.lexsort((gy, gx, self.pin_net))
        n_, x_, y_ = self.pin_net[order], gx[order], gy[order]
        keep = n_[1:] == n_[:-1]
        return (n_[:-1][keep], x_[:-1][keep], y_[:-1][keep], x_[1:][keep], y_[1:][keep])

    def route(self, cell_xy, macro_xy, seed=0):
        d, cfg, G = self.d, self.cfg, self.G
        rng = np.random.default_rng(seed)
        snet, x1, y1, x2, y2 = self._segments(cell_xy, macro_xy)
        S = len(x1)
        # ---- capacity: base tracks, minus macro blockage, minus local pin obstruction ----
        mcov = self.macro_density(macro_xy)
        pin = np.zeros((G, G))
        cgx = np.clip((cell_xy[:, 0] / self.gs).astype(int), 0, G - 1)
        cgy = np.clip((cell_xy[:, 1] / self.gs).astype(int), 0, G - 1)
        np.add.at(pin, (cgy, cgx), 1.0)
        pin = pin / max(pin.mean(), 1e-9)
        cap_h = self.base_cap * np.clip(1.0 - 0.92 * mcov, 0.02, 1.0) * np.clip(1.0 - 0.06 * pin, 0.35, 1.0)
        cap_v = (self.base_cap * np.clip(1.0 - 0.92 * mcov, 0.02, 1.0)
                 * np.clip(1.0 - 0.06 * pin, 0.35, 1.0))
        hist_h = np.zeros((G, G)); hist_v = np.zeros((G, G))
        uh = np.zeros((G, G)); uv = np.zeros((G, G))
        best = None

        def prefix(ch, cv):
            return (np.pad(np.cumsum(ch, axis=1), ((0, 0), (1, 0))),
                    np.pad(np.cumsum(cv, axis=0), ((1, 0), (0, 0))))

        lo_x, hi_x = np.minimum(x1, x2), np.maximum(x1, x2)
        lo_y, hi_y = np.minimum(y1, y2), np.maximum(y1, y2)
        xm = x2.copy(); ym = y2.copy(); useA = np.ones(S, bool)

        for it in range(cfg.route_iters):
            ch = (1.0 + hist_h) * (1.0 + 9.0 * np.clip(uh - cap_h, 0, None) / cap_h)
            cv = (1.0 + hist_v) * (1.0 + 9.0 * np.clip(uv - cap_v, 0, None) / cap_v)
            Sx, Sy = prefix(ch, cv)
            det = 1 + 2 * it                      # detour budget grows as negotiation proceeds
            # Two route families, both single-trunk:
            #   A: horizontal-vertical-horizontal, trunk column xm   (xm = x1 or x2 -> an L)
            #   B: vertical-horizontal-vertical,  trunk row    ym
            XM = np.stack([x1, x2, (x1 + x2) // 2,
                           rng.integers(np.maximum(lo_x - det, 0), np.minimum(hi_x + det, G - 1) + 1),
                           np.clip(lo_x - rng.integers(1, det + 2, S), 0, G - 1),
                           np.clip(hi_x + rng.integers(1, det + 2, S), 0, G - 1)], 1)
            YM = np.stack([y1, y2, (y1 + y2) // 2,
                           rng.integers(np.maximum(lo_y - det, 0), np.minimum(hi_y + det, G - 1) + 1),
                           np.clip(lo_y - rng.integers(1, det + 2, S), 0, G - 1),
                           np.clip(hi_y + rng.integers(1, det + 2, S), 0, G - 1)], 1)
            def hseg(row, a_, b_):                # cost of a horizontal run, prefix-sum lookup
                l, h_ = np.minimum(a_, b_), np.maximum(a_, b_)
                return Sx[row, h_ + 1] - Sx[row, l]
            def vseg(col, a_, b_):
                l, h_ = np.minimum(a_, b_), np.maximum(a_, b_)
                return Sy[h_ + 1, col] - Sy[l, col]
            cA = (hseg(y1[:, None], x1[:, None], XM) + hseg(y2[:, None], XM, x2[:, None]) +
                  vseg(XM, y1[:, None], y2[:, None]))
            cB = (vseg(x1[:, None], y1[:, None], YM) + vseg(x2[:, None], YM, y2[:, None]) +
                  hseg(YM, x1[:, None], x2[:, None]))
            iA, iB = np.argmin(cA, 1), np.argmin(cB, 1)
            xm, ym = XM[np.arange(S), iA], YM[np.arange(S), iB]
            useA = cA[np.arange(S), iA] <= cB[np.arange(S), iB]
            # ---- rip up everything, lay the new routes down (difference-array scatter) ---
            dh = np.zeros((G, G + 1)); dv = np.zeros((G + 1, G))
            def addh(row, a_, b_):
                l, h_ = np.minimum(a_, b_), np.maximum(a_, b_)
                np.add.at(dh, (row, l), 1.0); np.add.at(dh, (row, h_ + 1), -1.0)
            def addv(col, a_, b_):
                l, h_ = np.minimum(a_, b_), np.maximum(a_, b_)
                np.add.at(dv, (l, col), 1.0); np.add.at(dv, (h_ + 1, col), -1.0)
            A, B = useA, ~useA
            addh(y1[A], x1[A], xm[A]); addh(y2[A], xm[A], x2[A]); addv(xm[A], y1[A], y2[A])
            addv(x1[B], y1[B], ym[B]); addv(x2[B], ym[B], y2[B]); addh(ym[B], x1[B], x2[B])
            uh = np.cumsum(dh, axis=1)[:, :G]
            uv = np.cumsum(dv, axis=0)[:G, :]
            # ---- PathFinder history update: hist_g is the running Lagrange multiplier on
            #      GCell g's capacity constraint -- reused below as the congestion shadow price.
            hist_h += 0.45 * np.clip(uh - cap_h, 0, None) / cap_h
            hist_v += 0.45 * np.clip(uv - cap_v, 0, None) / cap_v
            tot_of = float(np.clip(uh - cap_h, 0, None).sum() + np.clip(uv - cap_v, 0, None).sum())
            if best is None or tot_of < best[0]:          # keep the best round, as real routers do
                best = (tot_of, uh.copy(), uv.copy(), xm.copy(), ym.copy(), useA.copy())

        _, uh, uv, xm, ym, useA = best
        # ---- per-net cost, for the "externality on everyone else" accounting -------------
        seg_cost = np.where(useA,
                            np.abs(x1 - xm) + np.abs(xm - x2) + np.abs(y1 - y2),
                            np.abs(y1 - ym) + np.abs(ym - y2) + np.abs(x1 - x2))
        lam = hist_h + hist_v
        mx = np.where(useA, xm, (x1 + x2) // 2); my = np.where(useA, (y1 + y2) // 2, ym)
        seg_pen = lam[y1, np.clip(mx, 0, G - 1)] + lam[np.clip(my, 0, G - 1), mx] + lam[y2, np.clip(mx, 0, G - 1)]
        net_cost = np.zeros(d.NNET)
        np.add.at(net_cost, snet, seg_cost * (1.0 + seg_pen))

        oh = np.clip(uh - cap_h, 0, None); ov = np.clip(uv - cap_v, 0, None)
        ovfl = oh + ov
        # DRC markers: a GCell is written up once per v_unit of overflow beyond a tolerance band,
        # which is how a signoff DRC deck turns continuous congestion into a countable violation.
        cap_t = cap_h + cap_v
        v_unit = 0.25 * self.base_cap
        hotspot = np.clip(ovfl - 0.10 * cap_t, 0, None) / v_unit
        return dict(uh=uh, uv=uv, cap_h=cap_h, cap_v=cap_v, ovfl=ovfl, hotspot=hotspot,
                    lam=lam, wl=float(seg_cost.sum()), net_cost=net_cost)

    # -- the oracle call ----------------------------------------------------------------
    def evaluate(self, macro_xy, noise=0.0) -> Result:
        macro_xy = np.asarray(macro_xy, float)
        cell_xy = self.place(macro_xy)
        seed = int(hashlib.md5(np.round(macro_xy, 3).tobytes()).hexdigest()[:8], 16)
        r = self.route(cell_xy, macro_xy, seed=seed)
        self.calls += 1
        drc = float(r["hotspot"].sum())
        if noise:                                             # optional tool nondeterminism
            drc *= float(np.random.default_rng(seed + 1).normal(1.0, noise))
        return Result(drc=drc, n_hotspot=int((r["hotspot"] >= 1.0).sum()), wl=r["wl"],
                      hotspot=r["hotspot"], ovfl=r["ovfl"], usage=r["uh"] + r["uv"],
                      cap=r["cap_h"] + r["cap_v"], lam=r["lam"], cell_xy=cell_xy,
                      net_cost=r["net_cost"], macro_xy=macro_xy.copy())


ORACLES = {k: Oracle(d, CFG) for k, d in DESIGNS.items()}
BASE: Dict[str, Result] = {}
for k, o in ORACLES.items():
    t0 = time.time()
    BASE[k] = o.evaluate(DESIGNS[k].base_sites)
    t = time.time() - t0
    nb = o.evaluate(DESIGNS[k].null_sites)
    print(f"{k:11s} base DRC={BASE[k].drc:8.1f} hotspot-GCells={BASE[k].n_hotspot:4d} "
          f"WL={BASE[k].wl:8.0f} | peripheral-'off' DRC={nb.drc:7.1f} | {t*1000:6.0f} ms/call")
ORACLE_SECONDS_PER_CALL = t

## 3. Visual toolkit

A matplotlib style, a bitmap die-shot renderer (PIL) and a small SVG writer. Figures are produced in four media on purpose: plots for the statistics, bitmaps for the layouts, vector documents for the verdicts, and an animation for the repair loop.

In [ ]:
# =====================================================================================
# 4. Visual toolkit: matplotlib style, a PIL die-shot renderer, SVG writer
# =====================================================================================
# Color assignment follows the job of the data: one hue, light->dark for magnitude
# (congestion); two hues + neutral midpoint for polarity (DRC deltas); a fixed
# categorical order for the attribution methods, never cycled.

SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7", "#e34948", "#008300"]
INK, INK2, INK3 = "#0b0b0b", "#52514e", "#8b8a84"
SURFACE, GRIDC = "#fcfcfb", "#e6e5e0"
METHOD_COLOR = {}   # filled in once the method list exists

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRIDC, "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2, "font.size": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.7,
    "axes.axisbelow": True, "legend.frameon": False, "figure.dpi": 130, "savefig.bbox": "tight",
})

CMAP_CONG = mcolors.LinearSegmentedColormap.from_list(
    "cong", ["#f7f9fc", "#cfe0f5", "#93bdec", "#4f92dd", "#2a78d6", "#1b4f8f", "#0d2745"])
CMAP_DIV = mcolors.LinearSegmentedColormap.from_list(
    "div", ["#1b4f8f", "#7fb0e4", "#eceae4", "#f0a184", "#c0392b"])


def save_fig(fig, name, also_pdf=True):
    fig.savefig(P("figures", name + ".png"), dpi=200)
    if also_pdf: fig.savefig(P("figures", name + ".pdf"))
    plt.close(fig)
    return P("figures", name + ".png")


# ---- fonts for the PIL renderer -------------------------------------------------------
def _font(sz, bold=False):
    base = os.path.join(matplotlib.get_data_path(), "fonts", "ttf")
    for f in (("DejaVuSans-Bold.ttf" if bold else "DejaVuSans.ttf"), "DejaVuSans.ttf"):
        try: return ImageFont.truetype(os.path.join(base, f), sz)
        except Exception: pass
    return ImageFont.load_default()


def _ramp_rgb(a, cmap, lo=None, hi=None):
    lo = np.nanmin(a) if lo is None else lo
    hi = np.nanmax(a) if hi is None else hi
    n = (a - lo) / max(hi - lo, 1e-9)
    return (np.asarray(cmap(np.clip(n, 0, 1)))[..., :3] * 255).astype(np.uint8)


def render_die(design: Design, res: Result, title: str, subtitle: str = "",
               field: str = "util", annot: Optional[Dict[int, str]] = None,
               highlight: Optional[Sequence[int]] = None, px: int = 1400) -> Image.Image:
    """A 'die shot': congestion raster + macro floorplan + DRC markers, drawn directly to a
    bitmap (no matplotlib), which is what a physical-design tool's GUI actually shows you."""
    pad_l, pad_t, pad_b = 60, 96, 74
    W = px + 2 * pad_l
    H = px + pad_t + pad_b
    img = Image.new("RGB", (W, H), SURFACE)
    dr = ImageDraw.Draw(img, "RGBA")
    G = res.usage.shape[0]

    fld = {"util": res.usage / np.maximum(res.cap, 1e-9), "hotspot": res.hotspot,
           "lam": res.lam, "ovfl": res.ovfl}[field]
    hi = float(np.percentile(fld, 99.5)) or 1.0
    heat = Image.fromarray(_ramp_rgb(fld[::-1], CMAP_CONG, 0.0, hi)).resize((px, px), Image.NEAREST)
    img.paste(heat, (pad_l, pad_t))

    sc = px / design.die
    fx = lambda x: pad_l + x * sc
    fy = lambda y: pad_t + px - y * sc

    # standard cells as a faint pin-density stipple
    step = max(1, len(res.cell_xy) // 2600)
    for x, y in res.cell_xy[::step]:
        dr.point((fx(x), fy(y)), fill=(255, 255, 255, 70))

    # DRC markers
    gs = design.die / G
    ys, xs = np.where(res.hotspot >= 1.0)
    for gy, gx in zip(ys, xs):
        cx, cy = fx((gx + .5) * gs), fy((gy + .5) * gs)
        r = 3.5
        dr.line([(cx - r, cy - r), (cx + r, cy + r)], fill="#e34948", width=2)
        dr.line([(cx - r, cy + r), (cx + r, cy - r)], fill="#e34948", width=2)

    # macros
    hl = set(highlight or [])
    for i in range(design.NM):
        w, h = design.macro_wh[i] * sc
        cx, cy = fx(res.macro_xy[i, 0]), fy(res.macro_xy[i, 1])
        box = [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]
        acc = "#e34948" if i in hl else "#0b0b0b"
        dr.rectangle(box, fill=(255, 255, 255, 205), outline=acc, width=5 if i in hl else 2)
        for k in range(1, 4):                                  # SRAM-ish hatch
            dr.line([box[0] + 4, box[1] + k * (h / 4), box[2] - 4, box[1] + k * (h / 4)],
                    fill=(11, 11, 11, 40), width=1)
        lab = design.macro_names[i] + (f"\n{annot[i]}" if annot and i in annot else "")
        f = _font(20, bold=True)
        dr.multiline_text((cx, cy), lab, font=f, fill=acc, anchor="mm", align="center")

    dr.rectangle([pad_l, pad_t, pad_l + px, pad_t + px], outline="#0b0b0b", width=2)
    dr.text((pad_l, 26), title, font=_font(34, True), fill=INK)
    dr.text((pad_l, 66), subtitle, font=_font(19), fill=INK2)

    # colorbar
    bx0, by0, bw, bh = pad_l, pad_t + px + 22, px * 0.45, 14
    ramp = Image.fromarray(_ramp_rgb(np.linspace(0, 1, 256)[None, :].repeat(8, 0), CMAP_CONG, 0, 1))
    img.paste(ramp.resize((int(bw), bh)), (int(bx0), int(by0)))
    dr.rectangle([bx0, by0, bx0 + bw, by0 + bh], outline=GRIDC)
    dr.text((bx0, by0 + bh + 6), {"util": "routing utilisation  0", "hotspot": "DRC markers  0",
                                  "lam": "congestion price  0", "ovfl": "overflow  0"}[field],
            font=_font(16), fill=INK2)
    dr.text((bx0 + bw, by0 + bh + 6), f"{hi:.2f}", font=_font(16), fill=INK2, anchor="ra")
    dr.line([(bx0 + bw + 60, by0 + 7 - 5), (bx0 + bw + 60 + 10, by0 + 7 + 5)], fill="#e34948", width=2)
    dr.line([(bx0 + bw + 60, by0 + 7 + 5), (bx0 + bw + 60 + 10, by0 + 7 - 5)], fill="#e34948", width=2)
    dr.text((bx0 + bw + 80, by0 + 2), "DRC marker", font=_font(16), fill=INK2)
    return img


def svg_open(w, h, bg=SURFACE):
    return [f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}" '
            f'viewBox="0 0 {w} {h}" font-family="DejaVu Sans, Helvetica, Arial, sans-serif">',
            f'<rect width="{w}" height="{h}" fill="{bg}"/>']


def svg_save(parts, name):
    parts.append("</svg>")
    path = P("svg", name + ".svg")
    with open(path, "w") as f: f.write("\n".join(parts))
    return path


ART: Dict[str, str] = {}      # registry of every artefact this notebook writes

for k, d in DESIGNS.items():
    im = render_die(d, BASE[k], f"{d.name}  -  as-given floorplan",
                    f"DRC markers {BASE[k].drc:.0f} in {BASE[k].n_hotspot} GCells   |   "
                    f"WL {BASE[k].wl:.0f}   |   {d.NM} macros, {d.NC} cell clusters, {d.NNET} nets")
    p = P("images", f"dieshot_{k}_base.png"); im.save(p); ART[f"dieshot_{k}"] = p
print("die shots:", [os.path.basename(v) for v in ART.values()])

## 4. The interventional dataset

A few hundred genuine `do()` operations per design: park a macro at its default site, relocate it inside an ECO radius, swap a pair, jitter a coalition. This is the ground truth, and the surrogates' training set.

In [ ]:
# =====================================================================================
# 5. The interventional dataset: a few hundred real do() operations per design
# =====================================================================================
# Each row is one *actual* place-and-route run under an intervention on the macro
# placement. This is the ground truth everything else is measured against, and the
# training set for the surrogates that amortise the 2^NM counterfactual queries.

try:
    from scipy import ndimage as ndi
    def _label(mask): return ndi.label(mask, structure=np.ones((3, 3)))
except Exception:                                              # pragma: no cover
    def _label(mask):
        lab = np.zeros(mask.shape, int); cur = 0
        for sy in range(mask.shape[0]):
            for sx in range(mask.shape[1]):
                if mask[sy, sx] and not lab[sy, sx]:
                    cur += 1; st = [(sy, sx)]; lab[sy, sx] = cur
                    while st:
                        y, x = st.pop()
                        for dy in (-1, 0, 1):
                            for dx in (-1, 0, 1):
                                ny, nx = y + dy, x + dx
                                if (0 <= ny < mask.shape[0] and 0 <= nx < mask.shape[1]
                                        and mask[ny, nx] and not lab[ny, nx]):
                                    lab[ny, nx] = cur; st.append((ny, nx))
        return lab, cur


def _dilate(mask, k=2):
    m = mask.copy()
    for _ in range(k):
        m = (m | np.roll(m, 1, 0) | np.roll(m, -1, 0) | np.roll(m, 1, 1) | np.roll(m, -1, 1))
    return m


def overlaps(xy, wh, i, j, gap=8.0):
    return (abs(xy[i, 0] - xy[j, 0]) < (wh[i, 0] + wh[j, 0]) / 2 + gap and
            abs(xy[i, 1] - xy[j, 1]) < (wh[i, 1] + wh[j, 1]) / 2 + gap)


def candidate_sites(d: Design, xy: np.ndarray, m: int, k: int = 24, rng=None, lattice=7,
                    radius: float = None):
    """Legal alternative sites for macro m, holding every other macro where it is. `radius`
    restricts to an ECO-sized local move, which is what a designer can actually afford late in
    the flow -- and it is what makes *which macro you move* matter."""
    rng = rng or np.random.default_rng(0)
    w, h = d.macro_wh[m]
    gx = np.linspace(w / 2 + 10, d.die - w / 2 - 10, lattice)
    gy = np.linspace(h / 2 + 10, d.die - h / 2 - 10, lattice)
    cand = np.array([(a, b) for a in gx for b in gy])
    cand = cand + rng.uniform(-8, 8, cand.shape)
    ok = []
    for c in cand:
        t = xy.copy(); t[m] = c
        if all(not overlaps(t, d.macro_wh, m, j) for j in range(d.NM) if j != m):
            ok.append(c)
    ok = np.array(ok) if ok else d.null_sites[m:m + 1]
    if radius is not None:
        near = ok[np.linalg.norm(ok - xy[m], axis=1) <= radius]
        ok = near if len(near) >= 3 else ok[np.argsort(np.linalg.norm(ok - xy[m], axis=1))[:6]]
    if len(ok) > k: ok = ok[rng.choice(len(ok), k, replace=False)]
    return ok


def _jitter(d, xy, m, rng, scale=0.10):
    for _ in range(30):
        c = xy[m] + rng.normal(0, scale * d.die, 2)
        c = np.clip(c, d.macro_wh[m] / 2 + 6, d.die - d.macro_wh[m] / 2 - 6)
        t = xy.copy(); t[m] = c
        if all(not overlaps(t, d.macro_wh, m, j) for j in range(d.NM) if j != m):
            return c
    return xy[m]


def build_interventions(name: str, n: int, seed: int):
    d, o = DESIGNS[name], ORACLES[name]
    rng = np.random.default_rng(seed)
    base, null = d.base_sites, d.null_sites
    rows, maps, XY = [], [], []

    def run(xy, kind, changed, offmask):
        r = o.evaluate(xy)
        rows.append(dict(design=name, kind=kind, drc=r.drc, n_hotspot=r.n_hotspot, wl=r.wl,
                         changed=",".join(map(str, changed)),
                         offmask="".join("1" if b else "0" for b in offmask)))
        maps.append(r.hotspot.astype(np.float16)); XY.append(xy.copy())
        return r

    run(base, "base", [], np.zeros(d.NM, bool))
    run(null, "all_off", list(range(d.NM)), np.ones(d.NM, bool))
    # (a) exhaustive single-macro "off": the but-for test, run for real on every macro
    for m in range(d.NM):
        xy = base.copy(); xy[m] = null[m]
        off = np.zeros(d.NM, bool); off[m] = True
        run(xy, "single_off", [m], off)
    # (a2) ECO-sized single-macro moves -- the action space the attributions must rank.
    #      These are *different draws* from the same radius as the evaluation candidate set,
    #      so the surrogate learns the local response surface without ever seeing the sites it
    #      will later be tested on.
    for m in range(d.NM):
        for _ in range(6 if not QUICK else 4):
            c = _jitter(d, base, m, rng, scale=0.09)
            xy = base.copy(); xy[m] = c
            run(xy, "eco_move", [m], np.zeros(d.NM, bool))
    # (b) exhaustive single-macro relocation to a lattice of legal sites
    for m in range(d.NM):
        for c in candidate_sites(d, base, m, k=4, rng=rng):
            xy = base.copy(); xy[m] = c
            run(xy, "single_move", [m], np.zeros(d.NM, bool))
    while len(rows) < n:
        u = rng.random()
        off = np.zeros(d.NM, bool)
        if u < 0.42:                                  # (c) coalition "off" -- the Shapley lattice
            k = int(rng.integers(1, d.NM + 1))
            S = rng.choice(d.NM, k, replace=False)
            off[S] = True
            xy = np.where(off[:, None], null, base)
            run(xy, "coalition_off", list(S), off)
        elif u < 0.72:                                # (d) coalition jitter
            k = int(rng.integers(1, max(2, d.NM // 2) + 1))
            S = rng.choice(d.NM, k, replace=False)
            xy = base.copy()
            for m in S: xy[m] = _jitter(d, xy, m, rng)
            run(xy, "coalition_jitter", list(S), off)
        elif u < 0.88:                                # (e) pairwise swap
            i, j = rng.choice(d.NM, 2, replace=False)
            xy = base.copy(); xy[[i, j]] = xy[[j, i]]
            if any(overlaps(xy, d.macro_wh, i, t) for t in range(d.NM) if t != i): continue
            run(xy, "swap", [int(i), int(j)], off)
        else:                                         # (f) mixed: some off, some relocated
            k = int(rng.integers(1, d.NM + 1))
            S = rng.choice(d.NM, k, replace=False)
            off[S[: max(1, len(S) // 2)]] = True
            xy = np.where(off[:, None], null, base)
            for m in S[max(1, len(S) // 2):]:
                xy[m] = candidate_sites(d, xy, m, k=1, rng=rng)[0]
            run(xy, "mixed", list(S), off)
    return pd.DataFrame(rows), np.stack(maps), np.stack(XY)


# The designer's actual action set late in the flow: an ECO-sized relocation of one macro.
# Every method is offered exactly this set, and the causal contrasts below are defined over it,
# because a cause you cannot act on is a cause you cannot use.
ECO_RADIUS = {n: 0.20 * DESIGNS[n].die for n in DESIGNS}
LOCAL_SITES = {n: {m: candidate_sites(DESIGNS[n], DESIGNS[n].base_sites, m,
                                      k=CFG.repair_candidates, lattice=11,
                                      rng=np.random.default_rng(CFG.seed + 41 + m),
                                      radius=ECO_RADIUS[n])
                   for m in range(DESIGNS[n].NM)} for n in DESIGNS}
print("ECO candidate sites per macro:",
      {n: [len(LOCAL_SITES[n][m]) for m in LOCAL_SITES[n]] for n in LOCAL_SITES})

DATA: Dict[str, dict] = {}
t0 = time.time()
for name in DESIGNS:
    df, mp, xy = build_interventions(name, CFG.n_interventions, CFG.seed + 91 * len(DATA))
    DATA[name] = dict(df=df, hot=mp, xy=xy)
    print(f"{name:11s} {len(df):5d} real P&R interventions   DRC "
          f"[{df.drc.min():.0f}, {df.drc.max():.0f}]  mean {df.drc.mean():.0f}   "
          f"({time.time() - t0:.0f}s elapsed)")

ALL_DF = pd.concat([v["df"] for v in DATA.values()], ignore_index=True)
ALL_DF.to_csv(P("data", "interventions.csv"), index=False)
ART["interventions_csv"] = P("data", "interventions.csv")
np.savez_compressed(P("data", "intervention_maps.npz"),
                    **{f"{k}_hot": v["hot"] for k, v in DATA.items()},
                    **{f"{k}_xy": v["xy"] for k, v in DATA.items()})
print(f"\ntotal oracle calls: {sum(o.calls for o in ORACLES.values())}   "
       f"wall clock: {time.time() - t0:.0f}s")

# ---- DRC hotspot regions: the specific 'violations' we will attribute blame for --------
REGIONS: Dict[str, List[dict]] = {}
for name, o in ORACLES.items():
    lab, ncc = _label(BASE[name].hotspot >= 1.0)
    mass = [(int((lab == i).sum()), float(BASE[name].hotspot[lab == i].sum()), i)
            for i in range(1, ncc + 1)]
    mass.sort(reverse=True)
    # The outcome we attribute blame for is "a violation still sits *here*", so the region is
    # dilated by two GCells: a hotspot that shifts by one GCell is the same hotspot.
    REGIONS[name] = [dict(rid=j, label=i, cells=c, drc=float(BASE[name].hotspot[_dilate(lab == i)].sum()),
                          core=(lab == i), mask=_dilate(lab == i))
                     for j, (c, m, i) in enumerate(mass[:3])]
    print(f"{name:11s} hotspot regions -> " +
          "  ".join(f"R{r['rid']}: {r['cells']} GCells / {r['drc']:.0f} markers"
                    for r in REGIONS[name]))

## 5. Surrogate A — U-Net on the congestion map

Predicts the DRC-marker map from the floorplan alone. It is also the vehicle the saliency baselines need: without a differentiable predictor there is nothing for Grad-CAM to be a gradient of.

In [ ]:
# =====================================================================================
# 6. Surrogate A -- a U-Net that predicts the DRC-marker map from the macro floorplan
# =====================================================================================
# Inputs are functions of the macro placement ONLY (no leakage from the oracle's own
# standard-cell placement or routing), so the network is a differentiable stand-in for the
# whole flow: floorplan -> placement -> routing -> DRC. That differentiability is exactly
# what the saliency baselines need, and it is what makes them look reasonable a priori.

def unet_inputs(name: str, macro_xy: np.ndarray) -> np.ndarray:
    d, o = DESIGNS[name], ORACLES[name]
    G, gs = o.G, o.gs
    cov = o.macro_density(macro_xy)
    covb = _gauss(cov, max(1, G // 10))
    deg = np.array([len(o.nets_of_macro[m]) for m in range(d.NM)], float)
    deg = deg / max(deg.max(), 1.0)
    pinf = np.zeros((G, G))
    yy, xx = np.mgrid[0:G, 0:G] + 0.5
    for m in range(d.NM):
        cx, cy = macro_xy[m] / gs
        s = max(1.5, (d.macro_wh[m].mean() / gs))
        pinf += deg[m] * np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * s * s))
    iof = np.zeros((G, G))
    for x, y in d.io_xy:
        iof[min(int(y / gs), G - 1), min(int(x / gs), G - 1)] += 1
    iof = _gauss(iof, max(1, G // 14)); iof /= max(iof.max(), 1e-9)
    free = _gauss(1.0 - cov, max(1, G // 8))
    free = free / max(free.mean(), 1e-9)
    rx = np.tile(np.linspace(-1, 1, G), (G, 1)); ry = rx.T
    return np.stack([cov, covb, pinf / max(pinf.max(), 1e-9), iof, free / 2.0, rx, ry]).astype(np.float32)


CH_IN = 7
X_ALL, Y_ALL, D_ALL = [], [], []
for name, v in DATA.items():
    for i in range(len(v["df"])):
        X_ALL.append(unet_inputs(name, v["xy"][i])); Y_ALL.append(v["hot"][i].astype(np.float32))
        D_ALL.append(name)
X_ALL = np.stack(X_ALL); Y_ALL = np.stack(Y_ALL); D_ALL = np.array(D_ALL)
Y_LOG = np.log1p(Y_ALL)
rng = np.random.default_rng(CFG.seed)
perm = rng.permutation(len(X_ALL))
n_val = max(12, int(0.18 * len(perm)))
VAL_IDX, TRN_IDX = perm[:n_val], perm[n_val:]
print(f"U-Net dataset: {X_ALL.shape}  train {len(TRN_IDX)}  val {len(VAL_IDX)}")


if HAS_TORCH:
    class Block(nn.Module):
        def __init__(s, i, o_):
            super().__init__()
            s.f = nn.Sequential(nn.Conv2d(i, o_, 3, padding=1), nn.GroupNorm(4, o_), nn.SiLU(),
                                nn.Conv2d(o_, o_, 3, padding=1), nn.GroupNorm(4, o_), nn.SiLU())
        def forward(s, x): return s.f(x)

    class UNet(nn.Module):
        """3-level U-Net; a scalar DRC head hangs off the bottleneck so the same network gives
        both the marker map and the total, and so d(DRC)/d(input) is well defined."""
        def __init__(s, cin=CH_IN, w=32):
            super().__init__()
            s.e1, s.e2, s.e3 = Block(cin, w), Block(w, 2 * w), Block(2 * w, 4 * w)
            s.d2, s.d1 = Block(4 * w + 2 * w, 2 * w), Block(2 * w + w, w)
            s.out = nn.Conv2d(w, 1, 1)
            s.head = nn.Sequential(nn.Linear(4 * w, 64), nn.SiLU(), nn.Linear(64, 1))
        def forward(s, x):
            e1 = s.e1(x); e2 = s.e2(F.avg_pool2d(e1, 2)); e3 = s.e3(F.avg_pool2d(e2, 2))
            u2 = s.d2(torch.cat([F.interpolate(e3, size=e2.shape[-2:], mode="nearest"), e2], 1))
            u1 = s.d1(torch.cat([F.interpolate(u2, size=e1.shape[-2:], mode="nearest"), e1], 1))
            m = F.softplus(s.out(u1))[:, 0]
            g = F.softplus(s.head(e3.mean((-1, -2))))[:, 0]
            return m, g

    def train_unet():
        net = UNet().to(DEV)
        opt = torch.optim.AdamW(net.parameters(), 3e-3, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, CFG.unet_epochs)
        Xt = torch.tensor(X_ALL, device=DEV); Yt = torch.tensor(Y_LOG, device=DEV)
        Gt = torch.log1p(torch.tensor(Y_ALL.sum((1, 2)), device=DEV))
        hist = []
        bs = 16
        for ep in range(CFG.unet_epochs):
            net.train(); idx = TRN_IDX[torch.randperm(len(TRN_IDX)).numpy()]
            tot = 0.0
            for b in range(0, len(idx), bs):
                j = idx[b:b + bs]
                m, g = net(Xt[j])
                loss = F.mse_loss(m, Yt[j]) + 0.35 * F.mse_loss(g, Gt[j])
                opt.zero_grad(); loss.backward(); opt.step(); tot += float(loss) * len(j)
            sch.step()
            net.eval()
            with torch.no_grad():
                m, g = net(Xt[VAL_IDX])
                vl = float(F.mse_loss(m, Yt[VAL_IDX])); vg = float(F.mse_loss(g, Gt[VAL_IDX]))
            hist.append((tot / len(idx), vl, vg))
            if ep % max(1, CFG.unet_epochs // 8) == 0 or ep == CFG.unet_epochs - 1:
                print(f"  ep {ep:3d}  train {hist[-1][0]:.4f}  val-map {vl:.4f}  val-drc {vg:.4f}")
        return net, np.array(hist)

    t0 = time.time(); UNET, UHIST = train_unet()
    print(f"U-Net trained in {time.time() - t0:.0f}s on {DEV}")

    @torch.no_grad()
    def unet_predict(name, macro_xy):
        x = torch.tensor(unet_inputs(name, macro_xy)[None], device=DEV)
        m, g = UNET(x)
        return np.expm1(m[0].cpu().numpy()), float(np.expm1(g.cpu().numpy()[0]))
else:                                                          # pragma: no cover
    UNET, UHIST = None, np.zeros((1, 3))
    _A = X_ALL.reshape(len(X_ALL), -1); _A = np.c_[_A, np.ones(len(_A))]
    _W = np.linalg.lstsq(_A[TRN_IDX], Y_LOG.reshape(len(X_ALL), -1)[TRN_IDX], rcond=1e-3)[0]
    def unet_predict(name, macro_xy):
        x = np.r_[unet_inputs(name, macro_xy).ravel(), 1.0]
        m = np.expm1(x @ _W).reshape(X_ALL.shape[-2:])
        return m, float(m.sum())

# ---- held-out fidelity, per design ----------------------------------------------------
def spearman(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ra = np.argsort(np.argsort(a)); rb = np.argsort(np.argsort(b))
    ra, rb = ra - ra.mean(), rb - rb.mean()
    return float((ra @ rb) / max(np.sqrt((ra @ ra) * (rb @ rb)), 1e-12))

UNET_FID = []
for name in DESIGNS:
    idx = [i for i in VAL_IDX if D_ALL[i] == name]
    true = Y_ALL[idx].sum((1, 2))
    pred = np.array([unet_predict(name, DATA[name]["xy"][j])[1]
                     for j in [list(np.where(D_ALL == name)[0]).index(i) for i in idx]])
    UNET_FID.append(dict(design=name, n=len(idx), spearman=spearman(true, pred),
                         mae=float(np.abs(true - pred).mean()),
                         rel_mae=float((np.abs(true - pred) / np.maximum(true, 1)).mean())))
    print(f"{name:11s} held-out DRC   Spearman {UNET_FID[-1]['spearman']:+.3f}   "
          f"MAE {UNET_FID[-1]['mae']:.1f} markers ({100 * UNET_FID[-1]['rel_mae']:.1f}%)")
pd.DataFrame(UNET_FID).to_csv(P("tables", "unet_fidelity.csv"), index=False)

## 6. Surrogate B — GNN on the macro graph

Amortises the set function v̂(configuration), which is what necessity, responsibility and Shapley all query thousands of times. Fourier position features matter here: a plain MLP is too smooth to feel an ECO-sized move, and the DRC response to one is not smooth at all.

In [ ]:
# =====================================================================================
# 7. Surrogate B -- a GNN over the macro graph: the amortised set-function v-hat
# =====================================================================================
# The causal quantities are functions on the lattice of macro configurations (2^NM corners,
# more once relocations are allowed). The oracle cannot be called that many times, so we fit
# v-hat(configuration) -> (total DRC, DRC inside each named hotspot region) on the real
# interventions and query *it* for the combinatorics -- then re-verify the top of every
# ranking with real oracle runs (section 11).

NMAX = max(d.NM for d in DESIGNS.values())
DESIGN_ID = {n: i for i, n in enumerate(DESIGNS)}

NETSHARE = {}
for name, o in ORACLES.items():
    d = DESIGNS[name]
    S = np.zeros((d.NM, d.NM))
    for i in range(d.NM):
        for j in range(d.NM):
            if i != j:
                S[i, j] = len(np.intersect1d(o.nets_of_macro[i], o.nets_of_macro[j]))
    S = S + (DESIGNS[name].macro_module[:, None] == DESIGNS[name].macro_module[None, :]) * 1.0
    np.fill_diagonal(S, 0)
    NETSHARE[name] = S / max(S.max(), 1.0)

MACRO_DEG = {n: np.array([len(ORACLES[n].nets_of_macro[m]) for m in range(DESIGNS[n].NM)], float)
             for n in DESIGNS}


def gnn_features(name: str, macro_xy: np.ndarray):
    """Node features and adjacency for one configuration. Pure numpy, ~40 us per call, which is
    what makes 10^5 counterfactual queries affordable."""
    d = DESIGNS[name]; NM = d.NM
    xy = macro_xy / d.die
    base = d.base_sites / d.die; null = d.null_sites / d.die
    deg = MACRO_DEG[name] / max(MACRO_DEG[name].max(), 1)
    off = (np.linalg.norm(macro_xy - d.null_sites, axis=1) < 1e-6).astype(float)
    dsp = xy - base
    ctr = np.linalg.norm(xy - 0.5, axis=1)
    wh = d.macro_wh / d.die
    # crowding: how much macro area sits within 0.2 die of me
    D = np.linalg.norm(xy[:, None] - xy[None, :], axis=-1) + np.eye(NM) * 9
    crowd = ((wh.prod(1)[None, :] * np.exp(-(D / 0.18) ** 2)).sum(1))
    # Fourier position features: a plain MLP is too smooth to feel a 10%-of-die ECO move,
    # and the DRC response to one is not smooth at all.
    freqs = np.array([1.0, 2.0, 4.0, 8.0])
    ff = np.concatenate([np.sin(2 * np.pi * xy[:, :, None] * freqs).reshape(NM, -1),
                         np.cos(2 * np.pi * xy[:, :, None] * freqs).reshape(NM, -1)], 1)
    f = np.concatenate([np.stack([xy[:, 0], xy[:, 1], wh[:, 0], wh[:, 1], deg, off, dsp[:, 0],
                                  dsp[:, 1], ctr, crowd, np.linalg.norm(xy - null, axis=1),
                                  np.full(NM, NM / NMAX), np.full(NM, d.NNET / 6000.0)], 1),
                        ff], 1)
    A = NETSHARE[name] + np.exp(-(D / 0.22) ** 2) * (1 - np.eye(NM))
    A = A / np.maximum(A.sum(1, keepdims=True), 1e-9)
    F_ = np.zeros((NMAX, f.shape[1]), np.float32); F_[:NM] = f
    A_ = np.zeros((NMAX, NMAX), np.float32); A_[:NM, :NM] = A
    M_ = np.zeros(NMAX, np.float32); M_[:NM] = 1
    return F_, A_, M_


NF = gnn_features(list(DESIGNS)[0], DESIGNS[list(DESIGNS)[0]].base_sites)[0].shape[1]

# ---- assemble the training tensors in the same row order as the U-Net dataset ----------
GF, GA, GM, GID, GY = [], [], [], [], []
for name, v in DATA.items():
    masks = [r["mask"] for r in REGIONS[name]]
    while len(masks) < 3: masks.append(np.zeros_like(BASE[name].hotspot, bool))
    for i in range(len(v["df"])):
        f, a, m = gnn_features(name, v["xy"][i])
        GF.append(f); GA.append(a); GM.append(m); GID.append(DESIGN_ID[name])
        h = v["hot"][i].astype(np.float32)
        GY.append([h.sum()] + [float(h[mk].sum()) for mk in masks])
GF, GA, GM = np.stack(GF), np.stack(GA), np.stack(GM)
GID = np.array(GID); GY = np.log1p(np.stack(GY))
GY_MU, GY_SD = GY[TRN_IDX].mean(0), GY[TRN_IDX].std(0) + 1e-6
GYn = (GY - GY_MU) / GY_SD
print(f"GNN dataset: nodes<= {NMAX}, features {NF}, samples {len(GF)}, targets {GY.shape[1]}")

if HAS_TORCH:
    class MacroGNN(nn.Module):
        def __init__(s, nf=NF, w=96, layers=3, nd=len(DESIGNS)):
            super().__init__()
            s.emb = nn.Embedding(nd, 8)
            s.inp = nn.Linear(nf + 8, w)
            s.mp = nn.ModuleList([nn.Sequential(nn.Linear(3 * w, w), nn.SiLU(), nn.Linear(w, w))
                                  for _ in range(layers)])
            s.rd = nn.Sequential(nn.Linear(2 * w, w), nn.SiLU(), nn.Linear(w, w), nn.SiLU(),
                                 nn.Linear(w, 4))
        def forward(s, f, a, m, did):
            e = s.emb(did)[:, None, :].expand(-1, f.shape[1], -1)
            h = torch.relu(s.inp(torch.cat([f, e], -1))) * m[..., None]
            for L in s.mp:
                agg = torch.bmm(a, h)
                glob = (h.sum(1) / m.sum(1, keepdim=True).clamp(min=1))[:, None].expand_as(h)
                h = (h + L(torch.cat([h, agg, glob], -1))) * m[..., None]
            pooled = torch.cat([h.sum(1) / m.sum(1, keepdim=True).clamp(min=1), h.max(1).values], -1)
            return s.rd(pooled)

    def train_gnn():
        net = MacroGNN().to(DEV)
        opt = torch.optim.AdamW(net.parameters(), 4e-3, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, CFG.gnn_epochs)
        f = torch.tensor(GF, device=DEV); a = torch.tensor(GA, device=DEV)
        m = torch.tensor(GM, device=DEV); did = torch.tensor(GID, device=DEV)
        y = torch.tensor(GYn, dtype=torch.float32, device=DEV)
        tr = torch.tensor(TRN_IDX, device=DEV); va = torch.tensor(VAL_IDX, device=DEV)
        hist = []
        for ep in range(CFG.gnn_epochs):
            net.train()
            p = net(f[tr], a[tr], m[tr], did[tr]); loss = F.mse_loss(p, y[tr])
            opt.zero_grad(); loss.backward(); opt.step(); sch.step()
            net.eval()
            with torch.no_grad():
                vl = float(F.mse_loss(net(f[va], a[va], m[va], did[va]), y[va]))
            hist.append((float(loss), vl))
            if ep % max(1, CFG.gnn_epochs // 6) == 0 or ep == CFG.gnn_epochs - 1:
                print(f"  ep {ep:4d}  train {float(loss):.4f}  val {vl:.4f}")
        return net, np.array(hist)

    t0 = time.time(); GNN, GHIST = train_gnn()
    print(f"GNN trained in {time.time() - t0:.0f}s")

    @torch.no_grad()
    def v_hat_batch(name, XYs) -> np.ndarray:
        """Amortised counterfactual: array of configurations -> (n, 4) predicted
        [total DRC, DRC in R0, R1, R2]."""
        out = []
        did = DESIGN_ID[name]
        for b in range(0, len(XYs), 512):
            chunk = XYs[b:b + 512]
            fs, as_, ms = zip(*[gnn_features(name, x) for x in chunk])
            p = GNN(torch.tensor(np.stack(fs), device=DEV), torch.tensor(np.stack(as_), device=DEV),
                    torch.tensor(np.stack(ms), device=DEV),
                    torch.full((len(chunk),), did, device=DEV, dtype=torch.long))
            out.append(p.cpu().numpy())
        return np.expm1(np.concatenate(out) * GY_SD + GY_MU)
else:                                                          # pragma: no cover
    GNN, GHIST = None, np.zeros((1, 2))
    _Z = np.concatenate([GF.reshape(len(GF), -1), GM], 1); _Z = np.c_[_Z, np.ones(len(_Z))]
    _WG = np.linalg.lstsq(_Z[TRN_IDX], GYn[TRN_IDX], rcond=1e-3)[0]
    def v_hat_batch(name, XYs):
        z = np.stack([np.r_[gnn_features(name, x)[0].ravel(), gnn_features(name, x)[2], 1.0]
                      for x in XYs])
        return np.expm1((z @ _WG) * GY_SD + GY_MU)

def v_hat(name, xy): return v_hat_batch(name, np.asarray(xy)[None])[0]

GNN_FID = []
for name in DESIGNS:
    idx = [i for i in VAL_IDX if D_ALL[i] == name]
    loc = [list(np.where(D_ALL == name)[0]).index(i) for i in idx]
    pred = v_hat_batch(name, DATA[name]["xy"][loc])[:, 0]
    true = np.expm1(GY[idx, 0])
    GNN_FID.append(dict(design=name, n=len(idx), spearman=spearman(true, pred),
                        mae=float(np.abs(true - pred).mean()),
                        rel_mae=float((np.abs(true - pred) / np.maximum(true, 1)).mean())))
    print(f"{name:11s} held-out DRC   Spearman {GNN_FID[-1]['spearman']:+.3f}   "
          f"MAE {GNN_FID[-1]['mae']:.1f} markers ({100 * GNN_FID[-1]['rel_mae']:.1f}%)")
pd.DataFrame(GNN_FID).to_csv(P("tables", "gnn_fidelity.csv"), index=False)
t0 = time.time(); _ = v_hat_batch(list(DESIGNS)[0], np.repeat(DESIGNS[list(DESIGNS)[0]].base_sites[None], 512, 0))
print(f"amortised query cost: {(time.time() - t0) / 512 * 1e6:.0f} us per counterfactual "
      f"vs {ORACLE_SECONDS_PER_CALL * 1e6:.0f} us for a real P&R run "
      f"({ORACLE_SECONDS_PER_CALL / ((time.time() - t0) / 512):.0f}x)")

## 7. Halpern–Pearl actual causation

Probability of necessity, probability of sufficiency, degree of responsibility (Chockler–Halpern) and blame — computed against the designer's real action set, because a cause you cannot act on is a cause you cannot use.

In [ ]:
# =====================================================================================
# 8. Halpern-Pearl actual causation: necessity, sufficiency, responsibility, blame
# =====================================================================================
# Causal model.  Endogenous variables X_1..X_NM, one per macro, ranging over sites.  The
# actual context is the as-given floorplan (X_m = base_m); the default/"off" setting is the
# canonical peripheral site (null_m).  The outcome phi is a *specific* violation:
#     phi_R  ==  "the DRC hotspot in region R survives", i.e. markers(R) >= tau * markers_base(R)
# and phi_all == "the block still fails signoff", i.e. total DRC >= tau * base.
# Everything below is a query on that model; the surrogate answers them, the oracle audits them.

# A violation "survives" if enough of its marker mass remains. The block-level outcome uses a
# signoff-budget threshold (still failing by most of its original violation count); a named
# hotspot uses a laxer one, because a hotspot that halves has, for the designer, been fixed.
TAU_ALL, TAU_REGION = 0.80, 0.55
K_EPI = 24 if not QUICK else 12  # epistemic samples (uncertainty over the designer's context)
OUTCOMES = ["all", "R0", "R1", "R2"]


def config_from_mask(name, on_mask):
    d = DESIGNS[name]
    return np.where(np.asarray(on_mask, bool)[:, None], d.base_sites, d.null_sites)


# X_m ranges over the sites the designer can actually put macro m at: its default peripheral
# slot, or any ECO-sized relocation. HP quantifies over the alternative settings of the
# variable, so this action set *is* the contrast set -- both for "would it have happened but
# for M3" and for the witness sets W.
NALT = 3
CONTRAST = {n: {m: np.concatenate([DESIGNS[n].null_sites[m][None], LOCAL_SITES[n][m][:NALT]])
                for m in range(DESIGNS[n].NM)} for n in DESIGNS}
ALT = {n: {m: CONTRAST[n][m][1:] for m in CONTRAST[n]} for n in DESIGNS}
BASE_OUT = {n: np.array([BASE[n].drc] + [float(BASE[n].hotspot[r["mask"]].sum())
                                         for r in REGIONS[n]] + [0.0] * 3)[:4] for n in DESIGNS}
THRESH = {n: np.array([TAU_ALL, TAU_REGION, TAU_REGION, TAU_REGION]) * BASE_OUT[n]
          for n in DESIGNS}


def phi(name, pred):                      # pred: (..., 4) -> boolean survival of each outcome
    return pred >= THRESH[name][None, :]


def epistemic_contexts(name, rng, k=K_EPI, jitter=0.035):
    """The designer does not know the context exactly: every macro could plausibly have been
    placed a little differently. Blame (Chockler-Halpern) averages responsibility over this."""
    d = DESIGNS[name]
    out = [d.base_sites.copy()]
    while len(out) < k:
        xy = d.base_sites + rng.normal(0, jitter * d.die, d.base_sites.shape)
        xy = np.clip(xy, d.macro_wh / 2 + 5, d.die - d.macro_wh / 2 - 5)
        if all(not overlaps(xy, d.macro_wh, i, j)
               for i in range(d.NM) for j in range(i + 1, d.NM)):
            out.append(xy)
    return np.stack(out)


# ------------------------------------------------------------------ PN / PS / PNS
def necessity_sufficiency(name):
    d = DESIGNS[name]
    rng = np.random.default_rng(CFG.seed + 5)
    ctx = epistemic_contexts(name, rng)                       # (K, NM, 2)
    p_on = v_hat_batch(name, ctx)                             # outcome with everything in place
    on_holds = phi(name, p_on)                                # (K, 4)
    PN = np.zeros((d.NM, 4)); PS = np.zeros((d.NM, 4)); PNS = np.zeros((d.NM, 4))
    for m in range(d.NM):
        # --- PN: in contexts where the violation is present, is there an available move for m
        #     that kills it?  (the minimum over m's action set -- necessity you can act on) ---
        p_off = np.min(np.stack([v_hat_batch(name, np.concatenate(
            [ctx[:, :m], np.repeat(c[None, None], len(ctx), 0), ctx[:, m + 1:]], 1))
            for c in CONTRAST[name][m]]), 0)
        killed = on_holds & ~phi(name, p_off)
        PN[m] = killed.sum(0) / np.maximum(on_holds.sum(0), 1)
        PNS[m] = killed.mean(0)
        # --- PS: start from clean coalitions (m off, violation absent), put m back ---------
        masks = rng.random((96, d.NM)) < 0.5
        masks[:, m] = False
        clean_xy = np.stack([config_from_mask(name, mk) for mk in masks])
        p_clean = v_hat_batch(name, clean_xy)
        is_clean = ~phi(name, p_clean)
        back = clean_xy.copy(); back[:, m] = d.base_sites[m]
        p_back = v_hat_batch(name, back)
        appears = is_clean & phi(name, p_back)
        PS[m] = appears.sum(0) / np.maximum(is_clean.sum(0), 1)
    return PN, PS, PNS


# ------------------------------------------- HP responsibility (Chockler & Halpern 2004)
def responsibility(name, xy_actual=None, kmax=None, budget=4000):
    """resp(m) = 1/(1+|W|) for the smallest witness set W of *other* macros whose relocation
    makes m pivotal: phi still holds under do(W=w) with m where it is (AC2(a)), and fails under
    do(W=w, m=x') for some site x' in m's action set (AC2(b)). |W| = 0 is plain but-for
    causation, so resp = 1; a macro that needs two other things to move first gets 1/3."""
    d = DESIGNS[name]
    kmax = CFG.resp_max_witness if kmax is None else kmax
    rng = np.random.default_rng(CFG.seed + 7)
    xy0 = d.base_sites if xy_actual is None else xy_actual
    settings = CONTRAST[name]
    resp = np.zeros((d.NM, 4)); frac = np.zeros((d.NM, 4)); wit = [[None] * 4 for _ in range(d.NM)]
    for m in range(d.NM):
        others = [j for j in range(d.NM) if j != m]
        done = np.zeros(4, bool)
        for k in range(0, kmax + 1):
            combos = list(itertools.combinations(others, k)) or [()]
            cfgs, meta = [], []
            for W in combos:
                choices = list(itertools.product(*[range(len(settings[j])) for j in W])) or [()]
                if len(choices) * len(combos) > budget:
                    choices = [choices[i] for i in rng.choice(len(choices),
                               min(len(choices), max(1, budget // max(len(combos), 1))), replace=False)]
                for ch in choices:
                    x = xy0.copy()
                    for j, c in zip(W, ch): x[j] = settings[j][c]
                    cfgs.append(x); meta.append((W, ch))
            cfgs = np.stack(cfgs)
            keep = phi(name, v_hat_batch(name, cfgs))                     # AC2(a): phi survives
            gone = np.zeros_like(keep)                                    # AC2(b): m is pivotal
            for c in settings[m]:                                         # under *some* move of m
                flip = cfgs.copy(); flip[:, m] = c
                gone |= ~phi(name, v_hat_batch(name, flip))
            piv = keep & gone
            for o in range(4):
                if not done[o] and piv[:, o].any():
                    i = int(np.argmax(piv[:, o]))
                    resp[m, o] = 1.0 / (1 + k); wit[m][o] = meta[i]; done[o] = True
                    # tie-break: HP responsibility is coarse (1, 1/2, 1/3, ...), so we also record
                    # how *robustly* m is pivotal -- the share of minimal witnesses that work.
                    frac[m, o] = float(piv[:, o].mean())
            if done.all(): break
    return resp, wit, frac


def blame(name, n_ctx=None):
    """Blame = E_context[responsibility], the epistemic average a court actually assigns."""
    d = DESIGNS[name]
    rng = np.random.default_rng(CFG.seed + 11)
    ctx = epistemic_contexts(name, rng, k=n_ctx or (8 if QUICK else 14))
    acc = np.zeros((d.NM, 4))
    for c in ctx:
        acc += responsibility(name, c, kmax=min(2, CFG.resp_max_witness), budget=1200)[0]
    return acc / len(ctx)


HP = {}
t0 = time.time()
for name in DESIGNS:
    PN, PS, PNS = necessity_sufficiency(name)
    R, W, RF = responsibility(name)
    B = blame(name)
    # ranking score: HP degree of responsibility, tie-broken by robustness of pivotality
    RS = R * (0.5 + 0.5 * RF)
    HP[name] = dict(PN=PN, PS=PS, PNS=PNS, resp=R, witness=W, blame=B, resp_frac=RF, resp_score=RS)
    top = np.argsort(-R[:, 0])[:3]
    print(f"{name:11s} ({time.time()-t0:5.0f}s)  most responsible for signoff failure: " +
          ", ".join(f"{DESIGNS[name].macro_names[m]} resp={R[m,0]:.2f} PN={PN[m,0]:.2f} "
                    f"PS={PS[m,0]:.2f} blame={B[m,0]:.2f}" for m in top))

## 8. Congestion pricing

The Pigouvian externality of each macro, measured exactly by intervention and estimated cheaply from the router's negotiated-congestion history, which is an estimate of the Lagrange multiplier on each GCell's capacity constraint.

In [ ]:
# =====================================================================================
# 9. Congestion pricing: what does this macro cost *everybody else*?
# =====================================================================================
# Transport economics has a name for the quantity a designer actually wants. A driver entering
# a congested road pays their own travel time but not the delay they add to everyone behind
# them; the Pigou/Vickrey congestion price is exactly that unpriced externality, and the LP
# dual of the capacity constraint is its shadow price. A macro is an agent on the routing
# graph: it consumes capacity (blockage) and it generates demand (its nets), and both push
# cost onto nets that have nothing to do with it.
#
#   (a) interventional externality   E_m = C_-m(m at its site) - C_-m(m at the default site)
#       -- C_-m sums routed cost over the nets NOT incident to m.  Vickrey, measured exactly.
#   (b) dual / shadow price          p_m = sum_g lambda_g * (capacity m removes at g)
#                                        + sum_g lambda_g * (demand m's own nets place at g)
#       -- lambda_g is the router's negotiated-congestion history cost, which is a running
#       estimate of the Lagrange multiplier on GCell g's capacity constraint (PathFinder).
#       One run, no interventions, and it comes with a *price field* over candidate sites.

def others_cost(name, res: Result, m: int) -> float:
    o = ORACLES[name]
    mask = np.ones(DESIGNS[name].NNET, bool)
    mask[o.nets_of_macro[m]] = False
    return float(res.net_cost[mask].sum())


def pigou_interventional(name) -> np.ndarray:
    """The exact Vickrey externality, one real P&R run per macro."""
    d, o = DESIGNS[name], ORACLES[name]
    E = np.zeros(d.NM)
    for m in range(d.NM):
        xy = d.base_sites.copy(); xy[m] = d.null_sites[m]
        r_off = o.evaluate(xy)
        E[m] = others_cost(name, BASE[name], m) - others_cost(name, r_off, m)
    return E


def price_dual(name, res: Result = None, macro_xy=None) -> np.ndarray:
    """First-order shadow price from a single run: no extra P&R calls."""
    d, o = DESIGNS[name], ORACLES[name]
    res = res or BASE[name]
    macro_xy = res.macro_xy if macro_xy is None else macro_xy
    lam = res.lam
    p = np.zeros(d.NM)
    for m in range(d.NM):
        blockage = float((lam * o.macro_cover_one(macro_xy, m)).sum()) * o.base_cap * 0.92 * 2
        demand = float(res.net_cost[o.nets_of_macro[m]].sum())
        p[m] = blockage + demand
    return p


def _net_anchor(name, res: Result, m: int):
    """Where m's own nets pull it: the centroid of every *other* pin on each incident net."""
    d, o = DESIGNS[name], ORACLES[name]
    pos = np.empty((len(o.pin_node), 2))
    pos[o.pin_is_cell] = res.cell_xy[o.pin_cell_id[o.pin_is_cell]]
    pos[o.pin_is_macro] = res.macro_xy[o.pin_macro_id[o.pin_is_macro]]
    pos[o.pin_is_io] = d.io_xy[o.pin_io_id[o.pin_is_io]]
    nets = o.nets_of_macro[m]
    anchors = []
    for n_ in nets:
        sel = (o.pin_net == n_) & ~(o.pin_is_macro & (o.pin_macro_id == m))
        if sel.any(): anchors.append(pos[sel].mean(0))
    return np.array(anchors) if anchors else res.macro_xy[m][None]


def price_field(name, m, res: Result = None, lattice=17):
    """Two first-order prices over every legal site for macro m, from a single routed run:

        externality(s) = sum_g lambda_g * (capacity m removes at g if placed at s)
        private(s)     = m's own nets' congestion-weighted length if m sits at s

    The *score* we attribute to a macro is the externality alone -- that is the Pigouvian
    quantity. The *action* the price recommends is argmin (private + externality): charge the
    agent for the congestion it imposes and let it choose, which is what a congestion charge
    does. Minimising the externality alone would just park every macro in a corner."""
    d, o = DESIGNS[name], ORACLES[name]
    res = res or BASE[name]
    sites = candidate_sites(d, res.macro_xy, m, k=10 ** 9, lattice=lattice)
    ext = np.array([float((res.lam * o.macro_cover_one(
        np.concatenate([res.macro_xy[:m], s[None], res.macro_xy[m + 1:]]), m)).sum())
        for s in sites]) * o.base_cap * 0.92 * 2
    anc = _net_anchor(name, res, m)
    mid = (sites[:, None, :] + anc[None, :, :]) / 2
    gi = np.clip((mid / o.gs).astype(int), 0, o.G - 1)
    lam_mid = res.lam[gi[..., 1], gi[..., 0]]
    dist = np.abs(sites[:, None, :] - anc[None, :, :]).sum(-1) / o.gs
    priv = (dist * (1.0 + lam_mid)).sum(1)
    return sites, ext, priv


PRICE = {}
for name in DESIGNS:
    Ei = pigou_interventional(name)
    Pd = price_dual(name)
    PRICE[name] = dict(pigou=Ei, dual=Pd)
    d = DESIGNS[name]
    order = np.argsort(-Ei)
    print(f"{name:11s} rho(interventional, dual) = {spearman(Ei, Pd):+.2f}   "
          f"priciest macros: " + ", ".join(f"{d.macro_names[m]} ({Ei[m]:+.0f})" for m in order[:3]))

# the price also tells you where to put it
PRICE_MOVE = {}
for name in DESIGNS:
    d = DESIGNS[name]
    mv = {}
    for m in range(d.NM):
        s, ext, priv = price_field(name, m)
        tot = ext + priv
        j = int(np.argmin(tot))
        at = np.argmin(np.abs(s - d.base_sites[m]).sum(1))     # the site nearest to where it is
        mv[m] = dict(site=s[j], gain=float(tot[at] - tot[j]), sites=s, ext=ext, priv=priv,
                     total=tot)
    PRICE_MOVE[name] = mv
    best = max(mv, key=lambda m: mv[m]["gain"])
    print(f"{name:11s} price gradient says: move {d.macro_names[best]} from "
          f"({d.base_sites[best,0]:.0f},{d.base_sites[best,1]:.0f}) to "
          f"({mv[best]['site'][0]:.0f},{mv[best]['site'][1]:.0f})  "
          f"[predicted price drop {mv[best]['gain']:.0f}]")

## 9. Shapley values

The cooperative-game reading: the unique attribution whose per-macro numbers sum to the block's total violation count.

In [ ]:
# =====================================================================================
# 10. Shapley values: the cooperative-game reading of the same question
# =====================================================================================
# v(S) = DRC when the macros in S sit at their as-given sites and the rest at the default.
# phi_m is m's Shapley value of v -- the unique attribution satisfying efficiency, symmetry,
# null-player and additivity, so the per-macro numbers add up to the total violation count.
# 2^NM is out of reach for the oracle; it is nothing for the surrogate (~45 us/query), which
# is precisely the division of labour this paper argues for.

def shapley(name, n_perm=None, outcome=0, antithetic=True):
    d = DESIGNS[name]
    n_perm = n_perm or CFG.shapley_perms
    rng = np.random.default_rng(CFG.seed + 23)
    phi_ = np.zeros(d.NM); cnt = 0
    B = 64
    while cnt < n_perm:
        perms = [rng.permutation(d.NM) for _ in range(B)]
        if antithetic: perms += [p[::-1] for p in perms]
        cfgs, tag = [], []
        for pi, p in enumerate(perms):
            on = np.zeros(d.NM, bool)
            cfgs.append(config_from_mask(name, on)); tag.append((pi, -1))
            for m in p:
                on[m] = True
                cfgs.append(config_from_mask(name, on)); tag.append((pi, int(m)))
        vals = v_hat_batch(name, np.stack(cfgs))[:, outcome]
        i = 0
        for pi, p in enumerate(perms):
            prev = vals[i]; i += 1
            for m in p:
                phi_[m] += vals[i] - prev; prev = vals[i]; i += 1
        cnt += len(perms)
    phi_ /= cnt
    return phi_, cnt


SHAP = {}
for name in DESIGNS:
    t0 = time.time()
    ph, npm = shapley(name)
    SHAP[name] = ph
    d = DESIGNS[name]
    tot = v_hat(name, d.base_sites)[0] - v_hat(name, d.null_sites)[0]
    print(f"{name:11s} {npm} permutations in {time.time()-t0:.1f}s  |  efficiency check: "
          f"sum(phi) {ph.sum():+.1f} vs v(N)-v(0) {tot:+.1f}  |  top: " +
          ", ".join(f"{d.macro_names[m]} {ph[m]:+.1f}" for m in np.argsort(-ph)[:3]))

# region-specific Shapley (blame for one named hotspot, not for the whole block)
SHAP_R = {n: np.stack([shapley(n, n_perm=max(400, CFG.shapley_perms // 3), outcome=1 + r)[0]
                       for r in range(3)], 1) for n in DESIGNS}
print("region-wise Shapley computed:", {n: SHAP_R[n].shape for n in SHAP_R})

## 10. Baselines: gradient saliency

What 'explainable ML for placement' usually means, implemented properly on the same surrogate so the comparison is fair.

In [ ]:
# =====================================================================================
# 11. The baselines we are arguing against: gradient saliency on the surrogate
# =====================================================================================
# This is what "explainable ML for placement" usually means: train a predictor of congestion,
# then ask which pixels it looked at. The maps are pretty and the question is correlational --
# a macro in a busy region lights up whether or not it is doing anything.

def macro_masks(name, macro_xy):
    o = ORACLES[name]
    return np.stack([o.macro_cover_one(macro_xy, m) for m in range(DESIGNS[name].NM)])


def _pool(sal, masks, mode="sum"):
    s = (sal[None] * masks).sum((1, 2))
    return s / masks.sum((1, 2)) if mode == "mean" else s


if HAS_TORCH:
    def gradcam(name, macro_xy):
        x = torch.tensor(unet_inputs(name, macro_xy)[None], device=DEV, requires_grad=True)
        feats = {}
        h = UNET.e3.register_forward_hook(lambda mod, i, o_: feats.__setitem__("e3", o_))
        _, g = UNET(x); h.remove()
        e3 = feats["e3"]
        gr = torch.autograd.grad(g.sum(), e3, retain_graph=False)[0]
        cam = F.relu((gr.mean((-1, -2), keepdim=True) * e3).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return cam[0, 0].detach().cpu().numpy()

    def input_grad(name, macro_xy, channel=0):
        x = torch.tensor(unet_inputs(name, macro_xy)[None], device=DEV, requires_grad=True)
        _, g = UNET(x)
        gr = torch.autograd.grad(g.sum(), x)[0][0, channel]
        return (gr * x[0, channel]).detach().cpu().numpy()

    def integrated_grad(name, macro_xy, steps=32, channel=0):
        x1 = torch.tensor(unet_inputs(name, macro_xy)[None], device=DEV)
        x0 = torch.tensor(unet_inputs(name, DESIGNS[name].null_sites)[None], device=DEV)
        acc = torch.zeros_like(x1)
        for a in np.linspace(1.0 / steps, 1.0, steps):
            xi = (x0 + a * (x1 - x0)).requires_grad_(True)
            _, g = UNET(xi)
            acc += torch.autograd.grad(g.sum(), xi)[0]
        ig = (acc / steps * (x1 - x0))[0, channel]
        return ig.detach().cpu().numpy()
else:                                                          # pragma: no cover
    def gradcam(name, macro_xy): return unet_predict(name, macro_xy)[0]
    def input_grad(name, macro_xy, channel=0):
        return _W[:-1, :].sum(1).reshape(CH_IN, *X_ALL.shape[-2:])[channel] * unet_inputs(name, macro_xy)[channel]
    def integrated_grad(name, macro_xy, steps=32, channel=0): return input_grad(name, macro_xy, channel)


def proximity_heuristic(name):
    """What a designer does today with the congestion map open: blame whatever macro is nearest
    the red. Purely correlational, and a surprisingly strong baseline."""
    d, o = DESIGNS[name], ORACLES[name]
    hs = BASE[name].hotspot
    yy, xx = np.mgrid[0:o.G, 0:o.G] + 0.5
    out = np.zeros(d.NM)
    for m in range(d.NM):
        cx, cy = d.base_sites[m] / o.gs
        s = max(2.0, d.macro_wh[m].mean() / o.gs)
        out[m] = float((hs * np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * (1.5 * s) ** 2))).sum())
    return out


SAL = {}
for name in DESIGNS:
    d = DESIGNS[name]
    mk = macro_masks(name, d.base_sites)
    cam = gradcam(name, d.base_sites)
    ig = integrated_grad(name, d.base_sites)
    xg = input_grad(name, d.base_sites)
    SAL[name] = dict(gradcam=_pool(cam, mk, "mean"), ig=_pool(ig, mk), inputgrad=_pool(xg, mk),
                     proximity=proximity_heuristic(name), cam_map=cam, ig_map=ig)
    print(f"{name:11s} Grad-CAM top: " +
          ", ".join(d.macro_names[m] for m in np.argsort(-SAL[name]['gradcam'])[:3]) +
          "   |  IG top: " + ", ".join(d.macro_names[m] for m in np.argsort(-SAL[name]['ig'])[:3]) +
          "   |  proximity top: " + ", ".join(d.macro_names[m] for m in np.argsort(-SAL[name]['proximity'])[:3]))

## 11. Ground truth and faithfulness

Ground truth is the tool. Every macro's but-for effect and its best available ECO move are measured by real re-runs, and every attribution is scored against them.

In [ ]:
# =====================================================================================
# 12. Ground truth and faithfulness
# =====================================================================================
# Ground truth is not a label file: it is the tool. For every macro we already ran the real
# but-for experiment (move it to the default site, re-place, re-route), so we know the true
# effect of each macro on the total violation count and on each named hotspot.

GT = {}
for name in DESIGNS:
    d = DESIGNS[name]; v = DATA[name]
    idx = {int(r.changed): i for i, r in v["df"].iterrows() if r.kind == "single_off"}
    dall = np.array([BASE[name].drc - v["df"].drc.values[idx[m]] for m in range(d.NM)])
    dreg = np.array([[float(BASE[name].hotspot[r["mask"]].sum()
                            - v["hot"][idx[m]].astype(np.float32)[r["mask"]].sum())
                      for r in REGIONS[name]] for m in range(d.NM)])
    GT[name] = dict(delta_all=dall, delta_region=dreg)
    print(f"{name:11s} true effect of removing each macro (DRC markers): " +
          "  ".join(f"{d.macro_names[m]} {dall[m]:+.0f}" for m in np.argsort(-dall)))

# The ECO ground truth: for every macro, the best *local* move the tool actually rewards.
# One real place-and-route run per candidate site -- expensive, and the only honest referee.
t0 = time.time()
for name in DESIGNS:
    d, o = DESIGNS[name], ORACLES[name]
    best = np.zeros(d.NM); bsite = np.zeros((d.NM, 2))
    for m in range(d.NM):
        vals = []
        for s in LOCAL_SITES[name][m]:
            xy = d.base_sites.copy(); xy[m] = s
            vals.append(o.evaluate(xy).drc)
        j = int(np.argmin(vals))
        best[m] = BASE[name].drc - vals[j]; bsite[m] = LOCAL_SITES[name][m][j]
    GT[name]["delta_local"] = best; GT[name]["best_site"] = bsite
    print(f"{name:11s} best single ECO move: {d.macro_names[int(np.argmax(best))]} "
          f"-> {100*best.max()/BASE[name].drc:.1f}% fewer DRC markers "
          f"(worst choice: {100*best.min()/BASE[name].drc:+.1f}%)")
print(f"ECO ground truth: {time.time()-t0:.0f}s\n")

def surrogate_effect(name):
    """The amortised but-for effect: how much would the surrogate's DRC drop if this macro made
    its best available ECO move? The simplest causal ranking there is, and the one the paper's
    surrogate is built to serve."""
    d = DESIGNS[name]
    b = v_hat(name, d.base_sites)[0]
    out = np.zeros(d.NM)
    for m in range(d.NM):
        cf = np.stack([np.concatenate([d.base_sites[:m], s[None], d.base_sites[m + 1:]])
                       for s in LOCAL_SITES[name][m]])
        out[m] = b - v_hat_batch(name, cf)[:, 0].min()
    return out


SURR_EFFECT = {n: surrogate_effect(n) for n in DESIGNS}
_rng = np.random.default_rng(CFG.seed + 31)
METHOD_SPEC = [
    # label,                group,       f(name) -> per-macro score,                 oracle calls at query time
    ("HP responsibility",  "causal",    lambda n: HP[n]["resp_score"][:, 0],         0),
    ("HP blame",           "causal",    lambda n: HP[n]["blame"][:, 0],              0),
    ("Prob. of necessity", "causal",    lambda n: HP[n]["PN"][:, 0],                 0),
    ("Shapley value",      "causal",    lambda n: SHAP[n],                           0),
    ("Counterfactual dDRC","causal",    lambda n: SURR_EFFECT[n],                    0),
    ("Pigou externality",  "economic",  lambda n: PRICE[n]["pigou"],                 lambda n: DESIGNS[n].NM),
    ("Shadow price",       "economic",  lambda n: PRICE[n]["dual"],                  0),
    ("Grad-CAM",           "saliency",  lambda n: SAL[n]["gradcam"],                 0),
    ("Integrated grads",   "saliency",  lambda n: SAL[n]["ig"],                      0),
    ("Input x gradient",   "saliency",  lambda n: SAL[n]["inputgrad"],               0),
    ("Proximity heuristic","heuristic", lambda n: SAL[n]["proximity"],               0),
    ("Random",             "control",   lambda n: _rng.random(DESIGNS[n].NM),        0),
]
METHODS = {m[0]: m[2] for m in METHOD_SPEC}
GROUP = {m[0]: m[1] for m in METHOD_SPEC}
GROUP_COLOR = {"causal": SERIES[0], "economic": SERIES[1], "saliency": SERIES[3],
               "heuristic": SERIES[4], "control": INK3}
METHOD_COLOR.update({m[0]: GROUP_COLOR[m[1]] for m in METHOD_SPEC})
SCORES = {n: {k: np.asarray(f(n), float) for k, f in METHODS.items()} for n in DESIGNS}

# ---- (i) rank agreement with the true but-for effects ---------------------------------
rows = []
for name in DESIGNS:
    for k in METHODS:
        rows.append(dict(design=name, method=k, group=GROUP[k],
                         rho_total=spearman(SCORES[name][k], GT[name]["delta_all"]),
                         rho_local=spearman(SCORES[name][k], GT[name]["delta_local"]),
                         rho_region=float(np.mean([spearman(SCORES[name][k],
                                                            GT[name]["delta_region"][:, r])
                                                   for r in range(len(REGIONS[name]))]))))
RANK_DF = pd.DataFrame(rows)
piv = RANK_DF.pivot_table(index="method", values=["rho_total", "rho_local", "rho_region"],
                          aggfunc="mean")
piv["group"] = [GROUP[m] for m in piv.index]
print("\nrank correlation with the tool's own but-for effects (mean over designs)")
print(piv.sort_values("rho_total", ascending=False).round(3).to_string())

# ---- (ii) deletion / insertion curves, measured with real P&R runs ---------------------
KDEL = min(5, min(d.NM for d in DESIGNS.values()))


def deletion_curve(name, order, insert=False):
    d, o = DESIGNS[name], ORACLES[name]
    out = []
    for k in range(0, KDEL + 1):
        on = np.ones(d.NM, bool) if not insert else np.zeros(d.NM, bool)
        on[list(order[:k])] = insert
        out.append(o.evaluate(config_from_mask(name, on)).drc)
    return np.array(out)


t0 = time.time()
DEL, INS = {}, {}
for name in DESIGNS:
    DEL[name], INS[name] = {}, {}
    for k in METHODS:
        order = np.argsort(-SCORES[name][k])
        DEL[name][k] = deletion_curve(name, order)
        INS[name][k] = deletion_curve(name, order, insert=True)
    # the best achievable curve given the same budget, by brute force over the true effects
    DEL[name]["_oracle_best"] = deletion_curve(name, np.argsort(-GT[name]["delta_all"]))
print(f"\ndeletion/insertion curves: {sum(o.calls for o in ORACLES.values())} cumulative oracle "
      f"calls, {time.time()-t0:.0f}s for this stage")

rows = []
for name in DESIGNS:
    b = BASE[name].drc
    for k in METHODS:
        c = DEL[name][k] / b
        rows.append(dict(design=name, method=k, group=GROUP[k],
                         drop_at_1=float(1 - c[1]), drop_at_3=float(1 - c[min(3, KDEL)]),
                         del_auc=float(1 - c[1:].mean()),
                         ins_auc=float((INS[name][k][1:] / b).mean())))
FAITH_DF = pd.DataFrame(rows)
FAITH = FAITH_DF.groupby(["method", "group"], as_index=False).mean(numeric_only=True)
print("\nfaithfulness (real re-runs): fraction of DRC removed by nulling the top-k macros")
print(FAITH.sort_values("del_auc", ascending=False).round(3).to_string(index=False))
RANK_DF.to_csv(P("tables", "rank_correlation.csv"), index=False)
FAITH_DF.to_csv(P("tables", "faithfulness.csv"), index=False)

## 12. Closed-loop validation — the table this is all for

Given a budget of macro moves, which macros should you move? Each method nominates; the tool grades.

In [ ]:
# =====================================================================================
# 13. Closed-loop validation: does the attribution actually fix the block?
# =====================================================================================
# The table the paper is for. A designer has a budget of k macro moves. Each method nominates
# which macros to move; the edit is applied; the real tool re-places and re-routes; we count
# the DRC markers that survive. Nobody grades their own homework -- the oracle does.
#
#   Protocol A (headline): every method uses the SAME site-selection rule, so the only thing
#     that varies is *which* macros were blamed. This isolates attribution quality.
#   Protocol B: each method also chooses the destination in its own idiom -- the congestion
#     price has a gradient, saliency only has "move it away from the bright pixels".

def _sites_for(name, xy, m, n=None):
    """One fixed ECO-sized candidate set per macro, shared by every method, so no method wins
    by being handed better destinations."""
    S = LOCAL_SITES[name][m]
    ok = np.array([s for s in S if all(not overlaps(np.concatenate([xy[:m], s[None], xy[m + 1:]]),
                                                    DESIGNS[name].macro_wh, m, j)
                                       for j in range(DESIGNS[name].NM) if j != m)])
    if not len(ok): ok = S[:1]
    return ok if n is None else ok[:n]


VERIFY_SITES = 3


def choose_site_surrogate(name, xy, m):
    """Common rule, and the loop the paper argues for: shortlist with the surrogate, then spend
    a fixed budget of REAL place-and-route runs to pick among the shortlist. Every method gets
    the same shortlist size and the same verification budget, so the only thing that varies
    between rows of the table is *which macro was blamed*."""
    S = _sites_for(name, xy, m)
    cfgs = np.stack([np.concatenate([xy[:m], s[None], xy[m + 1:]]) for s in S])
    top = np.argsort(v_hat_batch(name, cfgs)[:, 0])[:VERIFY_SITES]
    drc = [ORACLES[name].evaluate(cfgs[j]).drc for j in top]
    return S[int(top[int(np.argmin(drc))])]


def choose_site_price(name, xy, m, res):
    """The price-native edit: the same ECO candidate set, ranked by first-order social cost."""
    S = _sites_for(name, xy, m)
    o = ORACLES[name]
    ext = np.array([float((res.lam * o.macro_cover_one(
        np.concatenate([xy[:m], s[None], xy[m + 1:]]), m)).sum()) for s in S]) * o.base_cap * 1.84
    anc = _net_anchor(name, res, m)
    mid = (S[:, None, :] + anc[None, :, :]) / 2
    gi = np.clip((mid / o.gs).astype(int), 0, o.G - 1)
    priv = ((np.abs(S[:, None, :] - anc[None, :, :]).sum(-1) / o.gs)
            * (1.0 + res.lam[gi[..., 1], gi[..., 0]])).sum(1)
    return S[int(np.argmin(ext + priv))]


def choose_site_lowfield(name, xy, m, field):
    """The saliency-native edit: move the macro to where the map is coolest."""
    o = ORACLES[name]
    S = _sites_for(name, xy, m)
    load = [float((field * o.macro_cover_one(np.concatenate([xy[:m], s[None], xy[m + 1:]]), m)).sum())
            for s in S]
    return S[int(np.argmin(load))]


def repair(name, order, k, mode="surrogate", field=None):
    d, o = DESIGNS[name], ORACLES[name]
    xy = d.base_sites.copy()
    trace = [(None, o.evaluate(xy))]
    for m in order[:k]:
        m = int(m)
        if mode == "surrogate":  s = choose_site_surrogate(name, xy, m)
        elif mode == "price":    s = choose_site_price(name, xy, m, trace[-1][1])
        elif mode == "field":    s = choose_site_lowfield(name, xy, m, field)
        else:                    s = _sites_for(name, xy, m, 1)[0]
        xy = xy.copy(); xy[m] = s
        trace.append((m, o.evaluate(xy)))
    return xy, trace


NATIVE_MODE = {"Pigou externality": "price", "Shadow price": "price",
               "Grad-CAM": "field", "Integrated grads": "field", "Input x gradient": "field",
               "Proximity heuristic": "field", "Random": "random"}
NATIVE_FIELD = {"Grad-CAM": "cam_map", "Integrated grads": "ig_map"}

rows, TRACES = [], {}
t0 = time.time()
for name in DESIGNS:
    b_ = BASE[name]
    ranked = {k: [np.argsort(-SCORES[name][k])] for k in METHODS}
    # the control gets several draws, so it is not judged on one lucky order
    ranked["Random"] = [np.random.default_rng(CFG.seed + 500 + i).permutation(DESIGNS[name].NM)
                        for i in range(5)]
    ranked["Oracle-ranked (ground truth)"] = [np.argsort(-GT[name]["delta_local"])]
    for meth, orders in ranked.items():
        for rep, order in enumerate(orders):
            xyA, trA = repair(name, order, CFG.repair_topk, "surrogate")
            mode = NATIVE_MODE.get(meth, "surrogate")
            fld = (SAL[name][NATIVE_FIELD[meth]] if meth in NATIVE_FIELD else
                   (BASE[name].hotspot if mode == "field" else None))
            xyB, trB = repair(name, order, CFG.repair_topk, mode, fld)
            if rep == 0: TRACES[(name, meth)] = trA
            for k in range(1, CFG.repair_topk + 1):
                rA, rB = trA[k][1], trB[k][1]
                rows.append(dict(design=name, method=meth, group=GROUP.get(meth, "bound"),
                                 k=k, rep=rep,
                                 moved=",".join(DESIGNS[name].macro_names[int(m)] for m in order[:k]),
                                 drc_base=b_.drc, drc_A=rA.drc, drc_B=rB.drc,
                                 red_A=100 * (b_.drc - rA.drc) / b_.drc,
                                 red_B=100 * (b_.drc - rB.drc) / b_.drc,
                                 wl_A=100 * (rA.wl - b_.wl) / b_.wl,
                                 wl_B=100 * (rB.wl - b_.wl) / b_.wl))
REPAIR_DF = pd.DataFrame(rows)
_per_case = REPAIR_DF.groupby(["method", "group", "design", "k"], as_index=False).mean(numeric_only=True)
REPAIR = _per_case.groupby(["method", "group"], as_index=False).agg(
    red_A=("red_A", "mean"), red_B=("red_B", "mean"), wl_A=("wl_A", "mean"),
    wl_B=("wl_B", "mean"), worst=("red_A", "min"),
    win_rate=("red_A", lambda c: 100.0 * (np.asarray(c) > 0).mean())
).sort_values("red_A", ascending=False)
print(f"closed-loop repair: up to {CFG.repair_topk} macro moves per design, "
      f"{time.time()-t0:.0f}s, {sum(o.calls for o in ORACLES.values())} cumulative oracle calls\n")
print("DRC reduction after re-running place & route (%, mean over designs; higher is better)")
print(REPAIR.round(1).to_string(index=False))
REPAIR_DF.to_csv(P("tables", "repair_raw.csv"), index=False)
REPAIR.to_csv(P("tables", "repair_summary.csv"), index=False)

## 13. Figures

In [ ]:
# =====================================================================================
# 14. Figure pack (static): matplotlib panels, PIL die shots, hand-written SVG
# =====================================================================================
MSHORT = {"HP responsibility": "HP resp.", "Prob. of necessity": "PN", "HP blame": "blame",
          "Shapley value": "Shapley", "Counterfactual dDRC": "cf-dDRC",
          "Pigou externality": "Pigou", "Shadow price": "shadow price",
          "Integrated grads": "int. grads", "Input x gradient": "input x grad",
          "Proximity heuristic": "proximity", "Grad-CAM": "Grad-CAM", "Random": "random"}
ORDER = [m[0] for m in METHOD_SPEC]


def _norm(v):
    v = np.asarray(v, float); v = v - v.min()
    return v / max(v.max(), 1e-12)


# ---- F1: what the oracle produces ------------------------------------------------------
fig, axes = plt.subplots(len(DESIGNS), 3, figsize=(11.5, 3.6 * len(DESIGNS)))
for r, (name, res) in enumerate(BASE.items()):
    d, o = DESIGNS[name], ORACLES[name]
    ax = axes[r, 0]
    ax.scatter(res.cell_xy[:, 0], res.cell_xy[:, 1], s=.6, c="#c9d7ea", lw=0)
    for m in range(d.NM):
        w, h = d.macro_wh[m]
        ax.add_patch(plt.Rectangle(d.base_sites[m] - [w / 2, h / 2], w, h, fc="white",
                                   ec=INK, lw=1.1))
        ax.text(*d.base_sites[m], d.macro_names[m], ha="center", va="center", fontsize=7)
    ax.set(xlim=(0, d.die), ylim=(0, d.die), title=f"{name}: floorplan + placement",
           xticks=[], yticks=[]); ax.grid(False); ax.set_aspect(1)
    ax = axes[r, 1]
    im = ax.imshow(res.usage / res.cap, origin="lower", cmap=CMAP_CONG, vmin=0,
                   vmax=np.percentile(res.usage / res.cap, 99.5))
    plt.colorbar(im, ax=ax, fraction=.046).set_label("utilisation", color=INK2)
    ax.set(title="routing utilisation", xticks=[], yticks=[]); ax.grid(False)
    ax = axes[r, 2]
    im = ax.imshow(res.hotspot, origin="lower", cmap=CMAP_CONG, vmin=0)
    for reg in REGIONS[name]:
        ys, xs = np.where(reg["core"])
        ax.add_patch(plt.Rectangle((xs.min() - .5, ys.min() - .5), np.ptp(xs) + 1, np.ptp(ys) + 1,
                                   fill=False, ec=SERIES[1], lw=1.4))
        ax.text(xs.mean(), ys.max() + 1.5, f"R{reg['rid']}", color=SERIES[1], fontsize=8,
                ha="center")
    plt.colorbar(im, ax=ax, fraction=.046).set_label("DRC markers", color=INK2)
    ax.set(title=f"DRC markers ({res.drc:.0f} total)", xticks=[], yticks=[]); ax.grid(False)
fig.suptitle("The oracle: place, route, count violations, name the hotspots", y=1.005,
             fontsize=12, ha="center")
ART["fig1"] = save_fig(fig, "fig1_oracle")

# ---- F2: surrogate fidelity ------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.5))
ax = axes[0]
for i, (lab, h, c) in enumerate([("U-Net (map)", UHIST[:, 1] if len(UHIST) > 1 else [], SERIES[0]),
                                 ("GNN", GHIST[:, 1] if len(GHIST) > 1 else [], SERIES[2])]):
    if len(h): ax.plot(np.asarray(h) / max(np.asarray(h)[0], 1e-9), color=c, lw=2, label=lab)
ax.set(xlabel="epoch", ylabel="held-out loss (normalised)", title="surrogate training")
ax.legend()
for ax, (fid, lab, c) in zip(axes[1:], [(UNET_FID, "U-Net", SERIES[0]), (GNN_FID, "GNN", SERIES[2])]):
    for i, name in enumerate(DESIGNS):
        idx = [j for j in VAL_IDX if D_ALL[j] == name]
        loc = [list(np.where(D_ALL == name)[0]).index(j) for j in idx]
        true = Y_ALL[idx].sum((1, 2))
        pred = (np.array([unet_predict(name, DATA[name]["xy"][j])[1] for j in loc]) if lab == "U-Net"
                else v_hat_batch(name, DATA[name]["xy"][loc])[:, 0])
        ax.scatter(true, pred, s=14, color=SERIES[i], lw=0, alpha=.8, label=name)
    lim = [0, max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lim, lim, color=INK3, lw=1, ls="--")
    rho = np.mean([f["spearman"] for f in fid])
    ax.set(xlabel="true DRC markers (oracle)", ylabel=f"{lab} prediction",
           title=f"{lab}: held-out, mean rho = {rho:+.2f}")
    ax.legend(fontsize=7)
ART["fig2"] = save_fig(fig, "fig2_surrogates")

# ---- F3: the attributions disagree -----------------------------------------------------
fig, axes = plt.subplots(1, len(DESIGNS), figsize=(4.6 * len(DESIGNS), 4.6))
for ax, name in zip(np.atleast_1d(axes), DESIGNS):
    d = DESIGNS[name]
    M = np.stack([_norm(SCORES[name][k]) for k in ORDER])
    im = ax.imshow(M, cmap=CMAP_CONG, vmin=0, vmax=1, aspect="auto")
    ax.set(xticks=range(d.NM), xticklabels=d.macro_names, yticks=range(len(ORDER)),
           yticklabels=[MSHORT[k] for k in ORDER], title=f"{name}: who is to blame?")
    for i, k in enumerate(ORDER):
        ax.get_yticklabels()[i].set_color(METHOD_COLOR[k])
    ax.grid(False)
    j = int(np.argmax(GT[name]["delta_local"]))
    ax.add_patch(plt.Rectangle((j - .5, -.5), 1, len(ORDER), fill=False, ec=SERIES[1], lw=2.2))
    ax.text(j, len(ORDER) - .3, "true best fix", color=SERIES[1], ha="center", fontsize=8)
plt.colorbar(im, ax=np.atleast_1d(axes)[-1], fraction=.04).set_label("normalised score", color=INK2)
ART["fig3"] = save_fig(fig, "fig3_attribution_matrix")

# ---- F4: rank agreement with the tool's own counterfactuals ----------------------------
fig, ax = plt.subplots(figsize=(9.5, 4.2))
piv2 = RANK_DF.pivot_table(index="method", values=["rho_total", "rho_local", "rho_region"],
                           aggfunc="mean").reindex(ORDER)
x = np.arange(len(ORDER)); w = .27
targets = [("rho_total", "vs. total DRC effect", 1.0), ("rho_local", "vs. best ECO move", .72),
           ("rho_region", "vs. per-hotspot effect", .45)]
for i, (col, lab, alpha) in enumerate(targets):
    ax.bar(x + (i - 1) * w, piv2[col], w * .92,
           color=[METHOD_COLOR[m] for m in ORDER], alpha=alpha,
           edgecolor=SURFACE, linewidth=1.2)
# colour carries the method group, opacity carries the target -- so say so in neutral ink
handles = [plt.Rectangle((0, 0), 1, 1, fc=INK2, alpha=a, ec=SURFACE) for _, _, a in targets]
ax.axhline(0, color=INK2, lw=1)
ax.set(xticks=x, ylabel="Spearman rank correlation", title="Does the attribution agree with the tool?")
ax.set_xticklabels([MSHORT[m] for m in ORDER], rotation=32, ha="right")
for t, m in zip(ax.get_xticklabels(), ORDER): t.set_color(METHOD_COLOR[m])
ax.legend(handles, [t[1] for t in targets], ncol=3, fontsize=8)
ART["fig4"] = save_fig(fig, "fig4_rank_correlation")

# ---- F5: deletion curves, measured with real re-runs -----------------------------------
fig, axes = plt.subplots(1, len(DESIGNS) + 1, figsize=(3.6 * (len(DESIGNS) + 1), 3.4))
for ax, name in zip(axes, DESIGNS):
    for k in ORDER:
        ax.plot(DEL[name][k] / BASE[name].drc, color=METHOD_COLOR[k], lw=1.8,
                alpha=.95 if GROUP[k] in ("causal", "economic") else .55,
                ls="-" if GROUP[k] != "control" else ":")
    ax.plot(DEL[name]["_oracle_best"] / BASE[name].drc, color=INK, lw=1.6, ls="--")
    ax.set(title=name, xlabel="macros parked at default site", ylim=(0, 1.15),
           ylabel="DRC markers / base" if name == list(DESIGNS)[0] else None)
ax = axes[-1]
for k in ORDER:
    ax.plot(np.mean([DEL[n][k] / BASE[n].drc for n in DESIGNS], 0), color=METHOD_COLOR[k],
            lw=2.2, label=MSHORT[k], ls="-" if GROUP[k] != "control" else ":")
ax.plot(np.mean([DEL[n]["_oracle_best"] / BASE[n].drc for n in DESIGNS], 0), color=INK, lw=1.8,
        ls="--", label="oracle best")
ax.set(title="mean over designs", xlabel="macros parked at default site", ylim=(0, 1.15))
ax.legend(fontsize=6.5, ncol=2)
fig.suptitle("Park the accused: real place & route after removing the top-k blamed macros",
             y=1.02, fontsize=11)
ART["fig5"] = save_fig(fig, "fig5_deletion_curves")

# ---- F6: THE table, as a figure ---------------------------------------------------------
sub = REPAIR.set_index("method")
order6 = [m for m in ["Oracle-ranked (ground truth)"] + ORDER if m in sub.index]
fig, ax = plt.subplots(figsize=(9.5, 4.4))
vals = sub.loc[order6, "red_A"].values
cols = [INK if m == "Oracle-ranked (ground truth)" else METHOD_COLOR[m] for m in order6]
ax.bar(np.arange(len(order6)), vals, .68, color=cols, edgecolor=SURFACE, linewidth=1.2)
per = REPAIR_DF.groupby(["method", "design"], as_index=False).red_A.mean()
for i, m in enumerate(order6):
    v = per[per.method == m].red_A.values
    ax.scatter(np.full(len(v), i) + np.linspace(-.16, .16, len(v)), v, s=16,
               facecolor=SURFACE, edgecolor=INK2, lw=.9, zorder=3)
rnd = float(sub.loc["Random", "red_A"])
ax.axhline(rnd, color=INK3, lw=1.2, ls=":")
ax.text(len(order6) - .4, rnd + .6, "random control", color=INK3, fontsize=8, ha="right")
ax.axhline(0, color=INK2, lw=1)
ax.set(xticks=np.arange(len(order6)), ylabel="DRC markers removed (%)",
       title=f"Closed-loop repair: {CFG.repair_topk} ECO macro moves, verified by re-running the tool")
ax.set_xticklabels([MSHORT.get(m, "oracle-ranked\n(ground truth)") for m in order6],
                   rotation=32, ha="right")
for t, m in zip(ax.get_xticklabels(), order6):
    t.set_color(INK if m == "Oracle-ranked (ground truth)" else METHOD_COLOR[m])
ART["fig6"] = save_fig(fig, "fig6_repair_table")

# ---- F7: the money picture -- saliency looks elsewhere ----------------------------------
name = max(DESIGNS, key=lambda n: BASE[n].drc)
d = DESIGNS[name]
sal_top = int(np.argmax(SCORES[name]["Grad-CAM"]))
resp_top = int(np.argmax(SCORES[name]["HP responsibility"]))
gt_top = int(np.argmax(GT[name]["delta_local"]))
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (fld, title) in zip(axes, [(BASE[name].hotspot, "DRC markers (what failed)"),
                                   (SAL[name]["cam_map"], "Grad-CAM (what the model looked at)"),
                                   (BASE[name].lam, "congestion price lambda (what it costs)")]):
    ax.imshow(fld, origin="lower", cmap=CMAP_CONG)
    for m in range(d.NM):
        w, h = d.macro_wh[m] / ORACLES[name].gs
        c = d.base_sites[m] / ORACLES[name].gs
        hot = m in (sal_top, resp_top, gt_top)
        ax.add_patch(plt.Rectangle(c - [w / 2, h / 2], w, h, fill=False,
                                   ec=SERIES[1] if hot else "white", lw=2 if hot else .8))
        ax.text(*c, d.macro_names[m], color=SERIES[1] if hot else "white", fontsize=7,
                ha="center", va="center")
    ax.set(title=title, xticks=[], yticks=[]); ax.grid(False)
fig.suptitle(f"{name}: Grad-CAM blames {d.macro_names[sal_top]}, HP responsibility blames "
             f"{d.macro_names[resp_top]}, the tool says {d.macro_names[gt_top]}", y=1.03, fontsize=11)
ART["fig7"] = save_fig(fig, "fig7_saliency_vs_cause")

# ---- F8: the congestion price field and the move it recommends --------------------------
m = max(range(d.NM), key=lambda i: PRICE[name]["dual"][i])
S, ext, priv = price_field(name, m)
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2))
ax = axes[0]
ax.imshow(BASE[name].lam, origin="lower", cmap=CMAP_CONG,
          extent=[0, d.die, 0, d.die])
sc = ax.scatter(S[:, 0], S[:, 1], c=ext + priv, cmap="cividis", s=44, edgecolor="white", lw=.5)
j = int(np.argmin(ext + priv))
ax.annotate("", xy=S[j], xytext=d.base_sites[m],
            arrowprops=dict(arrowstyle="-|>", color=SERIES[1], lw=2.4))
ax.set(title=f"price field for {d.macro_names[m]} over its ECO sites", xticks=[], yticks=[])
ax.grid(False); plt.colorbar(sc, ax=ax, fraction=.046).set_label("private + external cost", color=INK2)
ax = axes[1]
o = np.argsort(ext + priv)
ax.plot(ext[o], color=SERIES[1], lw=2, label="externality (what it costs others)")
ax.plot(priv[o] * (ext.max() / max(priv.max(), 1e-9)), color=SERIES[0], lw=2,
        label="own routing cost (rescaled)")
ax.plot((ext + priv)[o] * (ext.max() / max((ext + priv).max(), 1e-9)), color=INK, lw=1.4, ls="--",
        label="social cost (rescaled)")
ax.set(xlabel="candidate site (sorted by social cost)", ylabel="first-order price",
       title="charge the externality, let the macro choose")
ax.legend(fontsize=8)
ART["fig8"] = save_fig(fig, "fig8_price_field")
print("static figures:", [k for k in ART if k.startswith("fig")])

## 14. Die shots, verdict cards, animation, interactive appendix

In [ ]:
# =====================================================================================
# 15. Media pack: die shots, a legal-style verdict card in SVG, an animated repair, Plotly
# =====================================================================================
# Not everything belongs in a matplotlib axis. A floorplan wants to be a bitmap; an argument
# about who is to blame wants to be a document; a repair wants to be a before/after.

# ---- (a) the verdict: PIL die shots, annotated with the attribution ---------------------
name = max(DESIGNS, key=lambda n: BASE[n].drc)
d, o = DESIGNS[name], ORACLES[name]
resp = HP[name]["resp_score"][:, 0]; pn = HP[name]["PN"][:, 0]
annot = {m: f"resp {HP[name]['resp'][m,0]:.2f}\nPN {pn[m]:.2f}" for m in range(d.NM)}
im = render_die(d, BASE[name], f"{name}  -  causal responsibility for signoff failure",
                "annotation: Halpern-Pearl degree of responsibility and probability of necessity",
                annot=annot, highlight=[int(np.argmax(resp))])
im.save(P("images", f"dieshot_{name}_responsibility.png"))
ART["dieshot_resp"] = P("images", f"dieshot_{name}_responsibility.png")

sal_top = int(np.argmax(SCORES[name]["Grad-CAM"]))
im2 = render_die(d, BASE[name], f"{name}  -  what gradient saliency looked at",
                 "annotation: Grad-CAM mass over each macro footprint (normalised)",
                 annot={m: f"cam {_norm(SCORES[name]['Grad-CAM'])[m]:.2f}" for m in range(d.NM)},
                 highlight=[sal_top], field="hotspot")
im2.save(P("images", f"dieshot_{name}_saliency.png"))
ART["dieshot_sal"] = P("images", f"dieshot_{name}_saliency.png")

# side-by-side contact sheet
sheet = Image.new("RGB", (im.width + im2.width, im.height), SURFACE)
sheet.paste(im, (0, 0)); sheet.paste(im2, (im.width, 0))
sheet.thumbnail((2400, 2400))
sheet.save(P("images", "contact_sheet_cause_vs_saliency.png"))
ART["contact_sheet"] = P("images", "contact_sheet_cause_vs_saliency.png")

# ---- (b) the repair, as an animation ---------------------------------------------------
best_method = REPAIR.iloc[0]["method"] if REPAIR.iloc[0]["method"] != "Oracle-ranked (ground truth)" \
    else REPAIR.iloc[1]["method"]
tr = TRACES[(name, best_method)]
frames = []
for i, (m, res) in enumerate(tr):
    hl = [int(x[0]) for x in tr[1:i + 1] if x[0] is not None]
    f = render_die(d, res, f"{name}  -  repair step {i}/{len(tr)-1}   ({best_method})",
                   ("as given" if m is None else
                    f"moved {d.macro_names[int(m)]}   ->   DRC {res.drc:.0f} "
                    f"({100*(res.drc-tr[0][1].drc)/tr[0][1].drc:+.1f}%)"),
                   highlight=hl, px=900)
    frames.append(f.convert("P", palette=Image.ADAPTIVE))
frames = frames + [frames[-1]] * 2
frames[0].save(P("anim", "repair_loop.gif"), save_all=True, append_images=frames[1:],
               duration=1100, loop=0)
ART["repair_gif"] = P("anim", "repair_loop.gif")

# ---- (c) SVG: the verdict card ----------------------------------------------------------
def verdict_card(name, outcome=0, path="verdict_card"):
    d = DESIGNS[name]
    R = HP[name]["resp"][:, outcome]; RS = HP[name]["resp_score"][:, outcome]
    PNv, PSv = HP[name]["PN"][:, outcome], HP[name]["PS"][:, outcome]
    pr = _norm(PRICE[name]["pigou"]); sh = SHAP[name]
    order = np.argsort(-RS)
    W, H = 980, 150 + 46 * d.NM
    s = svg_open(W, H)
    lbl = "signoff failure (total DRC)" if outcome == 0 else f"hotspot R{outcome-1}"
    s.append(f'<text x="34" y="52" font-size="27" font-weight="bold" fill="{INK}">'
             f'Attribution of responsibility &#8212; {name}</text>')
    s.append(f'<text x="34" y="78" font-size="15" fill="{INK2}">outcome under consideration: '
             f'{lbl}, {BASE[name].drc:.0f} markers in {BASE[name].n_hotspot} GCells</text>')
    cols = [("macro", 34), ("HP responsibility", 150), ("PN", 400), ("PS", 470),
            ("Shapley", 545), ("congestion price", 660), ("verdict", 800)]
    for t, x in cols:
        s.append(f'<text x="{x}" y="112" font-size="12.5" fill="{INK3}" '
                 f'letter-spacing="0.06em">{t.upper()}</text>')
    s.append(f'<line x1="34" y1="120" x2="{W-34}" y2="120" stroke="{GRIDC}" stroke-width="1.5"/>')
    for i, m in enumerate(order):
        y = 150 + 46 * i
        verdict = ("necessary cause" if PNv[m] > .6 else
                   "contributing cause" if RS[m] > .3 else
                   "sufficient, not necessary" if PSv[m] > .5 else "not a cause")
        vc = SERIES[7] if verdict == "necessary cause" else (
            SERIES[1] if verdict == "contributing cause" else INK3)
        s.append(f'<text x="34" y="{y+5}" font-size="17" font-weight="bold" fill="{INK}">'
                 f'{d.macro_names[m]}</text>')
        s.append(f'<rect x="150" y="{y-12}" width="{220*max(RS[m],0.004):.1f}" height="17" '
                 f'rx="4" fill="{SERIES[0]}"/>')
        s.append(f'<rect x="150" y="{y-12}" width="220" height="17" rx="4" fill="none" '
                 f'stroke="{GRIDC}"/>')
        s.append(f'<text x="378" y="{y+2}" font-size="12" fill="{INK2}" text-anchor="end">'
                 f'{R[m]:.2f}</text>')
        for x, v in ((400, PNv[m]), (470, PSv[m])):
            s.append(f'<text x="{x}" y="{y+2}" font-size="13" fill="{INK}">{v:.2f}</text>')
        s.append(f'<text x="545" y="{y+2}" font-size="13" fill="{INK}">{sh[m]:+.1f}</text>')
        s.append(f'<rect x="660" y="{y-10}" width="{110*pr[m]:.1f}" height="13" rx="3" '
                 f'fill="{SERIES[1]}" opacity="0.85"/>')
        s.append(f'<text x="800" y="{y+2}" font-size="13.5" fill="{vc}">{verdict}</text>')
        s.append(f'<line x1="34" y1="{y+22}" x2="{W-34}" y2="{y+22}" stroke="{GRIDC}"/>')
    s.append(f'<text x="34" y="{H-16}" font-size="11.5" fill="{INK3}">'
             f'PN = probability of necessity, PS = probability of sufficiency, both over the '
             f'designer\'s action set; responsibility = 1/(1+|W|) for the smallest witness '
             f'coalition W; price in units of routing cost imposed on other nets.</text>')
    return svg_save(s, path)


ART["verdict_svg"] = verdict_card(name)
for n_ in DESIGNS: verdict_card(n_, 0, f"verdict_card_{n_}")

# ---- (d) SVG: the causal model, drawn ---------------------------------------------------
def causal_diagram(name, m, path="causal_model"):
    d = DESIGNS[name]
    wit = HP[name]["witness"][m][0]
    W, H = 900, 400
    s = svg_open(W, H)
    s.append(f'<text x="30" y="44" font-size="23" font-weight="bold" fill="{INK}">'
             f'Why {d.macro_names[m]} is an actual cause</text>')
    s.append(f'<text x="30" y="70" font-size="14" fill="{INK2}">Halpern-Pearl AC2 with witness '
             f'set W = {{{", ".join(d.macro_names[j] for j in (wit[0] if wit else [])) or "empty"}}}'
             f'&#8194;&#8226;&#8194;degree of responsibility = '
             f'{HP[name]["resp"][m,0]:.2f}</text>')
    def node(x, y, t, fill, r=34):
        s.append(f'<circle cx="{x}" cy="{y}" r="{r}" fill="{fill}" stroke="{INK}" '
                 f'stroke-width="1.6"/>')
        s.append(f'<text x="{x}" y="{y+5}" font-size="14" font-weight="bold" text-anchor="middle" '
                 f'fill="{"white" if fill != SURFACE else INK}">{t}</text>')
    def arrow(x1, y1, x2, y2, c=INK2, dash=""):
        s.append(f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="{c}" stroke-width="2" '
                 f'marker-end="url(#a)" {dash}/>')
    s.append(f'<defs><marker id="a" markerWidth="9" markerHeight="9" refX="8" refY="3" '
             f'orient="auto"><path d="M0,0 L0,6 L8,3 z" fill="{INK2}"/></marker></defs>')
    node(110, 200, d.macro_names[m], SERIES[0])
    ys = np.linspace(130, 300, max(len(wit[0]) if wit else 0, 1))
    for j, yy in zip((wit[0] if wit else []), ys):
        node(110, yy + 90, d.macro_names[j], SERIES[3], 26)
        arrow(140, yy + 90, 320, 250)
    node(360, 200, "placer", SURFACE); node(560, 200, "router", SURFACE)
    node(770, 200, "DRC", SERIES[7])
    arrow(145, 200, 322, 200); arrow(396, 200, 522, 200); arrow(596, 200, 733, 200)
    s.append(f'<text x="30" y="360" font-size="13.5" fill="{INK2}">AC2(a): with W held at its '
             f'witness setting and {d.macro_names[m]} where it is, the violation still occurs. '
             f'AC2(b): move {d.macro_names[m]} alone and it does not. '
             f'Everything downstream &#8212; the standard cells, the routes &#8212; is recomputed '
             f'by the tool, which is what makes this a do() and not a perturbation.</text>')
    return svg_save(s, path)


ART["causal_svg"] = causal_diagram(name, int(np.argmax(HP[name]["resp_score"][:, 0])))

# ---- (e) Plotly: interactive congestion + method comparison -----------------------------
if HAS_PLOTLY:
    figs = []
    z = BASE[name].usage / BASE[name].cap
    f1 = go.Figure(go.Surface(z=z, colorscale="Blues", showscale=True,
                              colorbar=dict(title="utilisation")))
    f1.update_layout(title=f"{name}: routing utilisation surface (drag to rotate)",
                     scene=dict(zaxis_title="usage / capacity", xaxis_title="GCell x",
                                yaxis_title="GCell y"), height=620,
                     paper_bgcolor=SURFACE, font=dict(color=INK))
    figs.append(("congestion surface", f1))

    f2 = go.Figure()
    for k in ORDER:
        f2.add_trace(go.Bar(name=MSHORT[k], x=list(DESIGNS),
                            y=[REPAIR_DF[(REPAIR_DF.method == k) & (REPAIR_DF.design == n_)].red_A.mean()
                               for n_ in DESIGNS], marker_color=METHOD_COLOR[k]))
    f2.update_layout(barmode="group", title="DRC reduction after real re-runs, by design",
                     yaxis_title="% markers removed", height=520, paper_bgcolor=SURFACE,
                     plot_bgcolor=SURFACE, font=dict(color=INK))
    figs.append(("closed-loop repair", f2))

    dims = []
    for k in ORDER:
        col = RANK_DF[RANK_DF.method == k]
        dims.append(dict(label=MSHORT[k], values=[col.rho_total.mean(), col.rho_local.mean(),
                                                  col.rho_region.mean()]))
    f3 = go.Figure(go.Parcoords(line=dict(color=[0, 1, 2], colorscale="Blues"), dimensions=dims))
    f3.update_layout(title="rank correlation with ground truth (three targets)", height=460,
                     paper_bgcolor=SURFACE, font=dict(color=INK))
    figs.append(("attribution agreement", f3))

    parts = ['<meta charset="utf-8"><style>body{background:%s;color:%s;font-family:'
             'DejaVu Sans,Helvetica,Arial,sans-serif;max-width:1100px;margin:2rem auto}</style>'
             % (SURFACE, INK), "<h1>Interactive appendix</h1>"]
    for i, (t, f) in enumerate(figs):
        parts.append(f"<h2>{t}</h2>")
        parts.append(pio.to_html(f, include_plotlyjs="inline" if i == 0 else False,
                                 full_html=False))
    with open(P("html", "interactive.html"), "w") as fh: fh.write("\n".join(parts))
    ART["interactive"] = P("html", "interactive.html")
print("media:", {k: os.path.basename(v) for k, v in ART.items() if k not in ()})

## 15. The report

In [ ]:
# =====================================================================================
# 16. The report: one self-contained HTML file with every number and figure in it
# =====================================================================================
import base64


def b64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()


def img_tag(path, cap):
    ext = os.path.splitext(path)[1].lstrip(".")
    mime = {"png": "image/png", "gif": "image/gif", "jpg": "image/jpeg"}.get(ext, "image/png")
    return (f'<figure><img src="data:{mime};base64,{b64(path)}" alt="{cap}"/>'
            f'<figcaption>{cap}</figcaption></figure>')


def svg_inline(path, cap):
    with open(path) as f: s = f.read()
    return f'<figure><div class="svg">{s}</div><figcaption>{cap}</figcaption></figure>'


def table_html(df, caption, fmt="{:.3f}"):
    d2 = df.copy()
    for c in d2.columns:
        if d2[c].dtype.kind == "f": d2[c] = d2[c].map(lambda v: fmt.format(v))
    return (f'<figure><div class="tbl">{d2.to_html(index=False, border=0)}</div>'
            f'<figcaption>{caption}</figcaption></figure>')


def latex_table(df, name, caption):
    with open(P("tables", name + ".tex"), "w") as f:
        f.write(df.to_latex(index=False, float_format="%.2f", caption=caption,
                            label="tab:" + name))


latex_table(REPAIR.round(2), "repair_summary", "Closed-loop repair: DRC reduction after "
            "re-running place and route on the edits each attribution method recommends.")
latex_table(piv2.reset_index().round(3), "rank_correlation", "Rank correlation between each "
            "attribution and the tool's own counterfactual effects.")
latex_table(FAITH.round(3), "faithfulness", "Deletion/insertion faithfulness, measured by "
            "re-running place and route.")

# ---- read the findings off the numbers, so the prose cannot drift from the run ----------
GRP = lambda g: RANK_DF[RANK_DF.group.isin(g)].groupby("method").rho_total.mean()
diag_cause = GRP(["causal", "economic"]); diag_sal = GRP(["saliency"])
diag_heur = GRP(["heuristic"]); diag_rand = GRP(["control"])
rep = REPAIR.set_index("method")
rep_cause = rep[rep.group.isin(["causal", "economic"])].red_A
rep_sal = rep[rep.group == "saliency"].red_A
rep_heur = rep[rep.group == "heuristic"].red_A
bound = float(rep.loc["Oracle-ranked (ground truth)", "red_A"])
rnd_rep = float(rep.loc["Random", "red_A"])
FINDING = (
    f"<p><strong>On the diagnostic question &mdash; which macro is responsible for this "
    f"failure &mdash; the causal and economic attributions win outright.</strong> Their rank "
    f"correlation with the tool's own but-for effects averages "
    f"{diag_cause.mean():+.2f} (best: {diag_cause.idxmax()}, {diag_cause.max():+.2f}), against "
    f"{diag_sal.mean():+.2f} for gradient saliency (best {diag_sal.max():+.2f}, worst "
    f"{diag_sal.min():+.2f}) and {diag_rand.mean():+.2f} for the random control. Saliency is not "
    f"merely weaker here; on this target it carries almost no signal, which is what the "
    f"correlational/counterfactual distinction predicts.</p>"
    f"<p><strong>On the repair action, the picture is more interesting and less flattering to "
    f"a clean story.</strong> With a budget of {CFG.repair_topk} ECO moves, causal and economic "
    f"rankings remove {rep_cause.mean():.1f}% of DRC markers on average (best "
    f"{rep_cause.idxmax()} at {rep_cause.max():.1f}%) against {rnd_rep:.1f}% for the random "
    f"control and {rep_sal.mean():.1f}% for saliency &mdash; but the proximity heuristic, which "
    f"is simply <em>move whatever macro sits in the red</em>, reaches {rep_heur.max():.1f}%, "
    f"close to the {bound:.1f}% of the oracle-ranked bound. For an ECO-sized local move that is "
    f"not surprising: proximity is a decent proxy for <em>who can be relieved by moving a "
    f"little</em>, even though it is a poor proxy for <em>who is to blame</em> "
    f"({diag_heur.mean():+.2f} on the diagnostic target). The honest claim this run supports is "
    f"therefore narrower than 'causal beats everything': responsibility and price answer the "
    f"attribution question that saliency cannot answer at all, and they convert into repairs "
    f"that beat random and beat saliency &mdash; while a cheap spatial heuristic remains a "
    f"strong baseline for local repair specifically, and belongs in any honest table.</p>")

best = REPAIR[REPAIR.method != "Oracle-ranked (ground truth)"].iloc[0]
sal_best = REPAIR[REPAIR.group == "saliency"].red_A.max()
rnd = float(REPAIR[REPAIR.method == "Random"].red_A.iloc[0])
n_calls = sum(o.calls for o in ORACLES.values())

HERO = [("designs", f"{len(DESIGNS)}"),
        ("real P&R runs", f"{n_calls:,}"),
        ("counterfactuals answered", f"{len(DESIGNS) * CFG.shapley_perms * NMAX:,}"),
        ("best method", f"{MSHORT.get(best.method, best.method)}"),
        ("DRC removed", f"{best.red_A:.0f}%"),
        ("best saliency", f"{sal_best:.0f}%"),
        ("random control", f"{rnd:.0f}%")]

CSS = """
:root{--s:#fcfcfb;--ink:#0b0b0b;--ink2:#52514e;--ink3:#8b8a84;--line:#e6e5e0;--a:#2a78d6}
*{box-sizing:border-box}
body{margin:0;background:var(--s);color:var(--ink);font:16px/1.62 -apple-system,Segoe UI,
 "DejaVu Sans",Helvetica,Arial,sans-serif}
main{max-width:1080px;margin:0 auto;padding:3.5rem 1.4rem 6rem}
h1{font-size:2.35rem;line-height:1.15;letter-spacing:-.02em;margin:.2rem 0 .6rem}
h2{font-size:1.35rem;margin:3.2rem 0 .6rem;padding-top:1.2rem;border-top:1px solid var(--line)}
h3{font-size:1.02rem;color:var(--ink2);margin:1.8rem 0 .4rem}
p,li{color:var(--ink2);max-width:74ch}
strong{color:var(--ink)}
.lede{font-size:1.12rem;color:var(--ink2)}
.hero{display:flex;flex-wrap:wrap;gap:.5rem;margin:2rem 0}
.hero div{flex:1 1 130px;border:1px solid var(--line);border-radius:10px;padding:.8rem .9rem;
 background:#fff}
.hero b{display:block;font-size:1.5rem;color:var(--ink);letter-spacing:-.01em}
.hero span{font-size:.76rem;text-transform:uppercase;letter-spacing:.07em;color:var(--ink3)}
figure{margin:1.8rem 0}
figure img{width:100%;height:auto;border:1px solid var(--line);border-radius:8px;background:#fff}
figcaption{font-size:.86rem;color:var(--ink3);margin-top:.55rem}
.tbl{overflow-x:auto}
.svg{overflow-x:auto;border:1px solid var(--line);border-radius:8px;background:#fff;padding:.6rem}
.svg svg{max-width:100%;height:auto}
table{border-collapse:collapse;width:100%;font-size:.86rem;font-variant-numeric:tabular-nums}
th{text-align:left;font-weight:600;color:var(--ink3);text-transform:uppercase;font-size:.72rem;
 letter-spacing:.06em;border-bottom:1.5px solid var(--line);padding:.45rem .5rem}
td{padding:.42rem .5rem;border-bottom:1px solid var(--line);color:var(--ink2)}
tr:hover td{background:#f4f6f9}
code{background:#f1f0ec;padding:.1rem .3rem;border-radius:4px;font-size:.88em}
.tag{display:inline-block;font-size:.72rem;padding:.12rem .5rem;border-radius:99px;
 border:1px solid var(--line);color:var(--ink3);margin-right:.3rem}
"""

H = ["<title>Who's to Blame?</title>", f"<style>{CSS}</style>", "<main>",
     "<span class='tag'>reproduction notebook</span>"
     "<span class='tag'>actual causation</span><span class='tag'>congestion pricing</span>",
     "<h1>Who's to blame?</h1>",
     "<p class='lede'>Actual causation and congestion pricing for explaining macro-placement "
     "failures. Every number on this page was produced by re-running a real place-and-route "
     "flow under intervention &mdash; not by inspecting a model's gradients.</p>",
     "<div class='hero'>" + "".join(f"<div><span>{k}</span><b>{v}</b></div>" for k, v in HERO)
     + "</div>",
     "<h2>The question</h2>",
     "<p>Saliency answers <em>what did the model look at?</em> A designer asks the courtroom "
     "question: <strong>but for macro M3 being here, would this DRC hotspot exist?</strong> "
     "The Halpern&ndash;Pearl framework turns that into computable quantities &mdash; probability "
     "of necessity and sufficiency, and a degree of responsibility equal to "
     "1/(1+|W|) for the smallest coalition W of other changes that makes a macro pivotal. "
     "Transport economics supplies a second reading: each macro is an agent on the routing "
     "graph, and its Pigouvian congestion price is the routing cost it imposes on everyone "
     "else &mdash; an attribution denominated in cost, which turns directly into an action.</p>",
     "<h2>The oracle</h2>",
     f"<p>The interventional oracle is the flow itself: an analytical placer that re-places "
     f"{min(d.NC for d in DESIGNS.values())}&ndash;{max(d.NC for d in DESIGNS.values())} "
     f"standard-cell clusters in response to the macros, then a negotiated-congestion global "
     f"router whose overflow map is the DRC-marker proxy. One call costs "
     f"{ORACLE_SECONDS_PER_CALL*1000:.0f}&nbsp;ms, so a few hundred genuine <code>do()</code> "
     f"operations per design are affordable; the {len(ALL_DF):,} of them in this run are what "
     f"the surrogates are fitted to and what every claim below is checked against.</p>",
     img_tag(ART["fig1"], "The oracle. Left: macro floorplan and the placer's response. "
             "Middle: routing utilisation. Right: DRC markers, with the named hotspot regions "
             "R0-R2 that the causal analysis attributes blame for."),
     img_tag(ART["dieshot_ibex_like"], "A rendered die shot straight from the oracle "
             "(bitmap, not a plot): congestion raster, macro outlines, DRC markers."),
     "<h2>The surrogates</h2>",
     "<p>The causal quantities live on a lattice of macro configurations that no tool can "
     "enumerate. A U-Net predicts the marker map from the floorplan (and gives the saliency "
     "baselines something to be gradients of); a GNN over the macro graph amortises the "
     "set-function v&#770;(S) that necessity, responsibility and Shapley all query. The "
     "amortised query is roughly "
     f"{ORACLE_SECONDS_PER_CALL/45e-6:.0f}&times; cheaper than a real run.</p>",
     img_tag(ART["fig2"], "Held-out surrogate fidelity on interventions the models never saw."),
     "<h2>Six attributions, one floorplan</h2>",
     img_tag(ART["fig3"], "Normalised per-macro scores. The methods disagree, and the orange "
             "box marks the macro whose relocation the tool actually rewards most."),
     svg_inline(ART["verdict_svg"], "The attribution as a document: per-macro necessity, "
                "sufficiency, responsibility, Shapley share and congestion price, with the "
                "verdict each combination supports."),
     svg_inline(ART["causal_svg"], "Why the top-ranked macro qualifies as an actual cause under "
                "Halpern&ndash;Pearl AC2, and what its witness coalition is."),
     "<h2>Does it agree with the tool?</h2>",
     img_tag(ART["fig4"], "Rank correlation against three ground truths, all measured by "
             "re-running place and route: total DRC effect, best available ECO move, and "
             "per-hotspot effect."),
     img_tag(ART["fig5"], "Park the accused: DRC markers surviving after the top-k blamed "
             "macros are moved to their default sites and the block is re-placed and re-routed."),
     "<h2>What we found</h2>", FINDING,
     "<h2>The table this paper is for</h2>",
     "<p>A designer has a budget of ECO macro moves. Each method nominates which macros to "
     "move; every method is handed the same candidate sites and the same verification budget, "
     "so the only thing that varies is <strong>which macro was blamed</strong>. The tool grades "
     "the result.</p>",
     img_tag(ART["fig6"], "Closed-loop repair. Bars are means over designs; circles are the "
             "individual designs. The dotted line is a random-macro control averaged over five "
             "draws."),
     table_html(REPAIR.rename(columns={"red_A": "DRC removed % (shared sites)",
                                       "red_B": "DRC removed % (method-native sites)",
                                       "wl_A": "wirelength %", "worst": "worst design %",
                                       "win_rate": "cases improved %"})
                .drop(columns=["wl_B"]).round(2),
                "Closed-loop repair, full results.", "{:.1f}"),
     table_html(FAITH.round(3), "Deletion and insertion faithfulness (real re-runs)."),
     table_html(piv2.reset_index().round(3), "Rank correlation with the tool's counterfactuals."),
     "<h2>Why saliency fails here</h2>",
     img_tag(ART["fig7"], "The same block under three lenses. Grad-CAM highlights whatever sits "
             "in dense routing; responsibility asks whether removing it would have helped; the "
             "tool settles it."),
     img_tag(ART["contact_sheet"], "Left: responsibility and probability of necessity per macro. "
             "Right: Grad-CAM mass per macro. They do not name the same suspect."),
     "<h2>The price is an instruction</h2>",
     img_tag(ART["fig8"], "The congestion price field over one macro's legal ECO sites. The "
             "externality alone would park every macro in a corner; charging the externality and "
             "letting the macro minimise its own cost plus the charge gives the move."),
     "<h2>The repair, in motion</h2>",
     img_tag(ART["repair_gif"], "Each frame is a real place-and-route run after one more "
             "responsibility-ranked macro move."),
     "<h2>What would break this</h2>",
     "<ul>"
     "<li>The oracle here is a small placer and router, not OpenROAD. The pipeline is written "
     "against a one-method interface (<code>evaluate(macro_xy) -> Result</code>) and the final "
     "section ships the ORFS adapter, but the numbers on this page are the mini-flow's.</li>"
     "<li>Necessity and responsibility are computed against a <em>contrast set</em> &mdash; the "
     "sites a designer can actually move a macro to. Widen that set and the numbers move; "
     "a cause you cannot act on is not a useful cause.</li>"
     "<li>The surrogate is fitted per run on a few hundred interventions. Where it is wrong, "
     "the attributions are wrong, which is exactly why every headline claim is re-verified with "
     "real runs rather than read off the model.</li>"
     "<li>Three designs and a handful of ECO moves is a small sample, and multi-threaded CPU "
     "training is not bit-reproducible: repeated runs of this notebook move individual methods "
     "by a few points in the repair table, which is the same order as the gaps between "
     "neighbouring rows. The gaps that survive that noise are the ones between "
     "<em>groups</em> &mdash; causal/economic and the proximity heuristic above the random "
     "control, gradient saliency at or below it on the diagnostic target &mdash; and those are "
     "the only claims made here.</li></ul>",
     "</main>"]

with open(P("html", "report.html"), "w") as f: f.write("\n".join(H))
ART["report"] = P("html", "report.html")
print("report written:", ART["report"], f"({os.path.getsize(ART['report'])/1e6:.1f} MB)")

## 16. Reproducing against OpenROAD, and the summary

In [ ]:
# =====================================================================================
# 17. Running the identical experiments against OpenROAD / ORFS
# =====================================================================================
# Nothing above touches the mini-flow except through Oracle.evaluate(macro_xy) -> Result.
# On a machine with OpenROAD and the ORFS designs (ibex, aes, jpeg on Nangate45), drop this
# class in as ORACLES[name] and re-run from section 5: a few hundred interventions at a few
# minutes each is an overnight job on one CPU, which is the budget this method was designed for.

ORFS_ADAPTER = r'''
class ORFSOracle:
    """Interventional oracle backed by a real place-and-route flow.

    Requires: openroad on PATH, an ORFS design directory with the synthesised netlist, the
    Nangate45 platform files, and a floorplan DEF whose macros we overwrite per intervention.
    """

    TCL = """
    read_lef {lef}
    read_def {def_in}
    global_placement -density 0.68 -pad_left 2 -pad_right 2
    estimate_parasitics -placement
    global_route -congestion_report_file {rpt} -allow_congestion
    exit
    """

    def __init__(self, design, workdir, lef, def_in, grid=48, timeout=1800):
        import shutil
        assert shutil.which("openroad"), "openroad not on PATH"
        self.d, self.workdir, self.lef, self.def_in = design, workdir, lef, def_in
        self.G, self.timeout = grid, timeout
        os.makedirs(workdir, exist_ok=True)

    def _write_def(self, macro_xy, path):
        """Rewrite the COMPONENTS section: every macro gets FIXED at its intervened site.
        DEF is in database units; ORFS Nangate45 uses 2000 dbu/um."""
        dbu = 2000
        src = open(self.def_in).read()
        for i, nm in enumerate(self.d.macro_names):
            x = int((macro_xy[i, 0] - self.d.macro_wh[i, 0] / 2) * dbu)
            y = int((macro_xy[i, 1] - self.d.macro_wh[i, 1] / 2) * dbu)
            src = re.sub(rf"(- {re.escape(nm)} \S+\s*\+ )(FIXED|PLACED) \( [-\d]+ [-\d]+ \)",
                         rf"\g<1>FIXED ( {x} {y} )", src)
        open(path, "w").write(src)

    def evaluate(self, macro_xy):
        import subprocess, tempfile, re
        tag = hashlib.md5(np.round(macro_xy, 3).tobytes()).hexdigest()[:10]
        d_in = os.path.join(self.workdir, f"{tag}.def")
        rpt = os.path.join(self.workdir, f"{tag}.rpt")
        self._write_def(macro_xy, d_in)
        tcl = os.path.join(self.workdir, f"{tag}.tcl")
        open(tcl, "w").write(self.TCL.format(lef=self.lef, def_in=d_in, rpt=rpt))
        subprocess.run(["openroad", "-exit", tcl], check=True, timeout=self.timeout,
                       capture_output=True)
        # ORFS congestion report: one line per overflowing GCell edge
        H = np.zeros((self.G, self.G)); U = np.zeros((self.G, self.G)); C = np.ones((self.G, self.G))
        for line in open(rpt):
            m = re.match(r"\s*\(\s*(\d+)\s*,\s*(\d+)\s*\).*cap\s*=\s*(\d+).*usage\s*=\s*(\d+)", line)
            if m:
                gx, gy, cap, use = (int(v) for v in m.groups())
                if gx < self.G and gy < self.G:
                    U[gy, gx] += use; C[gy, gx] += cap
        ovfl = np.clip(U - C, 0, None)
        hot = np.clip(ovfl - 0.10 * C, 0, None) / max(C.mean() * 0.25, 1e-9)
        return Result(drc=float(hot.sum()), n_hotspot=int((hot >= 1).sum()), wl=float(U.sum()),
                      hotspot=hot, ovfl=ovfl, usage=U, cap=C, lam=ovfl / np.maximum(C, 1),
                      cell_xy=np.zeros((1, 2)), net_cost=np.zeros(self.d.NNET),
                      macro_xy=macro_xy.copy())
'''
with open(P("data", "orfs_adapter.py"), "w") as f:
    f.write("# Drop-in replacement for the mini-flow oracle.\nimport os, re, hashlib\n"
            "import numpy as np\n" + ORFS_ADAPTER)
ART["orfs_adapter"] = P("data", "orfs_adapter.py")
print(textwrap.dedent("""
    To reproduce against real silicon flows:
      1. build ORFS, run `make DESIGN_CONFIG=./designs/nangate45/ibex/config.mk` up to floorplan
      2. point ORFSOracle at results/nangate45/ibex/base/2_floorplan.def and the merged LEF
      3. ORACLES['ibex'] = ORFSOracle(design, workdir, lef, def_in); re-run from section 5
    The only thing that changes is the cost of a call: minutes instead of milliseconds, which is
    why the surrogate exists.
    """).strip())

# =====================================================================================
#  Summary
# =====================================================================================
best = REPAIR[REPAIR.method != "Oracle-ranked (ground truth)"].iloc[0]
sal = REPAIR[REPAIR.group == "saliency"]
caus = REPAIR[REPAIR.group.isin(["causal", "economic"])]
heur = REPAIR[REPAIR.group == "heuristic"]
diag = RANK_DF.groupby("group").rho_total.mean()
print("\n" + "=" * 86)
print("RESULT".center(86))
print("=" * 86)
print(f"  real place & route runs           : {sum(o.calls for o in ORACLES.values()):,}")
print(f"  surrogate counterfactual queries  : ~{len(DESIGNS)*CFG.shapley_perms*NMAX:,}")
print("  -- diagnosis: rank correlation with the tool's own but-for effects --")
for g in ["causal", "economic", "heuristic", "saliency", "control"]:
    print(f"    {g:10s} : {diag.get(g, float('nan')):+.3f}")
print("  -- repair: DRC removed after real re-runs --")
print(f"  best attribution                  : {best.method}  ({best.red_A:.1f}% DRC removed)")
print(f"  causal + economic methods (mean)  : {caus.red_A.mean():.1f}% DRC removed")
print(f"  gradient saliency (mean)          : {sal.red_A.mean():.1f}% DRC removed")
print(f"  proximity heuristic               : {heur.red_A.mean():.1f}% DRC removed")
print(f"  random control                    : {float(REPAIR[REPAIR.method=='Random'].red_A.iloc[0]):.1f}%")
print(f"  oracle-ranked ground truth        : "
      f"{float(REPAIR[REPAIR.method=='Oracle-ranked (ground truth)'].red_A.iloc[0]):.1f}%")
print("=" * 86)

INDEX = pd.DataFrame([dict(artifact=k, path=os.path.relpath(v, OUT),
                           kb=round(os.path.getsize(v) / 1024, 1)) for k, v in ART.items()]
                     + [dict(artifact=os.path.splitext(f)[0], path=os.path.join(sub, f),
                             kb=round(os.path.getsize(P(sub, f)) / 1024, 1))
                        for sub in ("tables", "figures", "svg", "images", "html", "data", "anim")
                        for f in sorted(os.listdir(P(sub)))]).drop_duplicates("path")
INDEX.to_csv(P("artifact_index.csv"), index=False)
print(f"\n{len(INDEX)} artifacts written under {os.path.abspath(OUT)}:")
print(INDEX.to_string(index=False))